# Assignment 2 – Natural-Language Video Retrieval Using MSR-VTT
## Text-to-Video Retrieval Using Semantic and Temporal Video Representations

**Course:** Multimodal Information Retrieval (AIMLZG543) · **Environment:** BITS Virtual Lab · **Total marks:** 10

| Group No. | Member | BITS ID | Contribution |
|---|---|---|---|
| 10 | TILAK | 2025AE05975 | System design, data pipeline, CLIP features, FAISS retrieval, lab execution |
| | OM PRAKASH UNIYAL | 2025AE05324 | Temporal models (pooling, Transformer, LSTM) and research ablation study |
| | KIRAN P | 2025AE05409 | Evaluation metrics, significance tests and results analysis |
| | GAURISHA MATHUR | 2025AE05642 | Streamlit dashboard and query edge-case handling |
| | KARAN RUPANI | 2025AE05813 | Experiments write-up and technical report |

**Task.** Build an end-to-end text→video retrieval system over MSR-VTT: index of **N = 100 clips** (5 clips × 20 categories),
**Q = 200 queries** (2 captions per clip) → **200 × 100 search matrix**, served through a Streamlit dashboard, evaluated with
standard IR metrics, and used to answer **Research Investigation 1**: *is mean-pooling frame embeddings sufficient, or does
explicit temporal modelling give superior retrieval?*

### How this notebook is organised
| Section | Rubric component |
|---|---|
| 0. Environment check | – |
| 1. Source code (all modules, written to disk with `%%writefile`) | all |
| 2–4. Configuration, data ingestion, frame feature extraction | **1. Data ingestion & feature engineering (2.0)** |
| 5–6. Learned temporal heads, vector index | 1 & 2 |
| 7–8. X-CLIP reference, evaluation | **4. Quantitative evaluation (1.5)** |
| 9. Research Investigation 1 | **5. Experimental analysis (2.0)** |
| 10–12. Edge cases, live text→video and video→text retrieval demos | **2. Cross-modal retrieval engine (2.5)** |
| 13–14. Efficiency, technical report | 4 & 5 |
| 15. Streamlit dashboard launched from the notebook | **3. Streamlit dashboard (2.0)** |

> **Run:** `Kernel → Restart & Run All`. All steps cache their outputs (`data/`, `artifacts/`), and frame features are
> saved after every clip, so if the lab session is interrupted, simply `Restart & Run All` again – it resumes where it stopped.
> Use the kernel **"Python (mmir-venv)"** – see the README for the one-time kernel registration.

## 0. Environment check

In [ ]:
# Verify the interpreter and that every required library is importable (versions are printed for reproducibility).
import sys, os, glob, platform, importlib
# If this kernel is the lab's default Python, also use the packages installed in ~/workspace/mmir-venv.
for sp in glob.glob(os.path.expanduser("~/workspace/mmir-venv/lib/python3*/site-packages")):
    if sp not in sys.path:
        sys.path.insert(0, sp)
        os.environ["PYTHONPATH"] = sp + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Python", sys.version.split()[0], "|", platform.platform())
for pkg in ["torch", "transformers", "huggingface_hub", "numpy", "pandas", "scipy", "cv2", "faiss",
            "streamlit", "altair", "matplotlib", "docx"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg:16s} {getattr(m, '__version__', 'ok')}")
    except Exception as e:
        print(f"  {pkg:16s} MISSING -> {e}")
import torch
print("CUDA available:", torch.cuda.is_available())

## 1. Source code

All application code is embedded below and written to disk with `%%writefile`, so the notebook is the single source of
truth. The package `mvr/` holds the retrieval engine; `app.py` is the Streamlit dashboard.

| Module | Responsibility |
|---|---|
| `mvr/config.py` | All settings (dataset, model, frames, training) – also editable from the dashboard |
| `mvr/data.py` | MSR-VTT download, stratified 5×20 sampling, OpenCV frame decoding, thumbnails |
| `mvr/encoders.py` | Frozen CLIP image/text encoders → shared 512-d space; optional X-CLIP |
| `mvr/temporal.py` | Mean/max/single/query-attentive pooling + learned temporal heads and their training |
| `mvr/index.py` | FAISS `IndexFlatIP` vector index (cosine similarity on unit vectors) |
| `mvr/query.py` | Query normalisation and edge cases (empty, typos, short, long, non-English) |
| `mvr/metrics.py` | R@K, MRR, mAP, nDCG@10, MdR, MnR + bootstrap / Wilcoxon / Holm |
| `mvr/control.py` | Positive-control experiment (synthetic order-only task) |
| `mvr/analysis.py` | Data-driven conclusions for Research Investigation 1 |
| `mvr/figures.py` | Report figures |
| `mvr/pipeline.py` | Orchestration with caching + all experiments |
| `mvr/report.py` | Generates the technical report (.docx) from the results |
| `app.py` | Streamlit dashboard |

In [ ]:
import os
os.makedirs('mvr', exist_ok=True)
os.makedirs('tests', exist_ok=True)
print('working directory:', os.getcwd())

#### `mvr/__init__.py`
Package marker.

In [ ]:
%%writefile mvr/__init__.py
"""mvr - MSR-VTT natural-language Video Retrieval (Assignment 2, Multimodal IR)."""
__version__ = "1.0.0"

#### `mvr/config.py`
**Configuration** – dataset protocol (5 clips × 20 categories, 2 captions/clip), backbone, frame count, training hyper-parameters, cached-output paths.

In [ ]:
%%writefile mvr/config.py
"""Central configuration for the MSR-VTT text-to-video retrieval system.

Every value can be overridden from the Streamlit "Pipeline Runner" page or via
environment variables (MVR_<NAME>), so the same code runs on CPU or GPU and
online (Hugging Face) or offline (manually placed files).
"""
from __future__ import annotations

import json
import os
from dataclasses import asdict, dataclass
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent

# Canonical MSR-VTT category names (Xu et al., CVPR 2016), id -> name.
MSRVTT_CATEGORIES = [
    "music", "people", "gaming", "sports/actions", "news/events/politics",
    "education", "tv shows", "movie/comedy", "animation", "vehicles/autos",
    "how-to", "travel", "science/technology", "animals/pets", "kids/family",
    "documentary", "food/drink", "cooking", "beauty/fashion", "advertisement",
]

# Methods evaluated. Order = display order.
PARAM_FREE_METHODS = ["single_frame", "mean_pool", "max_pool", "query_attn"]
LEARNED_METHODS = ["mlp_adapter", "set_tfm", "temporal_tfm", "seq_lstm"]

METHOD_INFO = {
    "single_frame": ("Single middle frame", "No temporal information (lower bound)."),
    "mean_pool": ("Mean pooling (baseline)", "Average of L2-normalised CLIP frame embeddings - order-agnostic."),
    "max_pool": ("Max pooling", "Element-wise max over frame embeddings - order-agnostic."),
    "query_attn": ("Query-attentive pooling", "Text-conditioned softmax over per-frame similarities (parameter-free, order-agnostic)."),
    "mlp_adapter": ("MLP adapter (control)", "Trained residual MLP on the mean-pooled vector; same data/loss as the temporal models but NO inter-frame modelling."),
    "set_tfm": ("Set Transformer (no positions)", "Trained self-attention across frames WITHOUT positional embeddings - models frame interactions but not order."),
    "temporal_tfm": ("Temporal Transformer (+positions)", "Trained self-attention across frames WITH learned temporal position embeddings (CLIP4Clip seqTransf-style) - explicit temporal order."),
    "seq_lstm": ("Temporal LSTM", "Trained uni-directional LSTM over the ordered frame sequence (CLIP4Clip seqLSTM-style)."),
    "xclip": ("X-CLIP (pretrained video model)", "Off-the-shelf spatio-temporal video-language model (cross-frame attention + multi-frame integration transformer)."),
}

ALL_SEARCH_METHODS = PARAM_FREE_METHODS + LEARNED_METHODS


@dataclass
class Config:
    # --- data ---
    hf_dataset_repo: str = "friedrichor/MSR-VTT"
    local_repo_dir: str = ""          # offline mirror of the HF repo layout (optional)
    data_dir: str = str(PROJECT_ROOT / "data" / "msrvtt")
    artifacts_dir: str = str(PROJECT_ROOT / "artifacts")
    videos_per_category: int = 5      # 20 categories x 5 = 100 index clips
    captions_per_video: int = 2       # 100 x 2 = 200 queries
    train_videos_per_category: int = 40  # disjoint training pool for learned temporal heads
    seed: int = 42

    # --- features ---
    clip_model: str = "openai/clip-vit-base-patch32"
    num_frames: int = 16              # uniformly sampled (segment mid-points)
    batch_size: int = 64
    device: str = "auto"              # auto | cuda | cpu

    # --- learned temporal heads ---
    train_seeds: tuple = (0, 1, 2)
    epochs: int = 40
    lr: float = 1e-4
    weight_decay: float = 0.05
    train_batch: int = 128
    tfm_layers: int = 2
    tfm_heads: int = 8

    cpu_fast: bool = True             # on CPU-only machines cap training size/epochs (see eff_* below)

    # --- optional external spatio-temporal model ---
    use_xclip: bool = True
    xclip_model: str = "microsoft/xclip-base-patch32"

    # --- evaluation ---
    bootstrap_samples: int = 10000
    ks: tuple = (1, 5, 10)

    # derived paths -------------------------------------------------------
    @property
    def videos_dir(self) -> Path:
        return Path(self.data_dir) / "videos"

    @property
    def subset_path(self) -> Path:
        return Path(self.data_dir) / "subset.json"

    @property
    def model_tag(self) -> str:
        return self.clip_model.rstrip("/").split("/")[-1].replace(".", "_")

    @property
    def features_path(self) -> Path:
        return Path(self.artifacts_dir) / f"features_{self.model_tag}_T{self.num_frames}.npz"

    @property
    def index_dir(self) -> Path:
        return Path(self.artifacts_dir) / f"index_{self.model_tag}_T{self.num_frames}"

    @property
    def models_dir(self) -> Path:
        return Path(self.artifacts_dir) / f"temporal_heads_{self.model_tag}"

    @property
    def results_path(self) -> Path:
        return Path(self.artifacts_dir) / f"results_{self.model_tag}_T{self.num_frames}.json"

    @property
    def figures_dir(self) -> Path:
        return Path(self.artifacts_dir) / "figures"

    @property
    def thumbs_dir(self) -> Path:
        return Path(self.artifacts_dir) / "thumbs"

    def resolved_device(self) -> str:
        if self.device != "auto":
            return self.device
        try:
            import torch
            return "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            return "cpu"

    def eff_train_per_cat(self) -> int:
        """Training clips per category actually used (capped at 25 on CPU when cpu_fast)."""
        if self.cpu_fast and self.resolved_device() == "cpu":
            return min(self.train_videos_per_category, 25)
        return self.train_videos_per_category

    def eff_epochs(self) -> int:
        if self.cpu_fast and self.resolved_device() == "cpu":
            return min(self.epochs, 20)
        return self.epochs

    def to_dict(self) -> dict:
        d = asdict(self)
        d["train_seeds"] = list(self.train_seeds)
        d["ks"] = list(self.ks)
        return d

    # persistence so the dashboard remembers the last-used settings
    def save(self, path: Path | None = None) -> None:
        path = path or Path(self.artifacts_dir) / "config.json"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(self.to_dict(), indent=2))

    @classmethod
    def load(cls, path: Path | None = None) -> "Config":
        cfg = cls()
        path = path or Path(os.environ.get("MVR_ARTIFACTS_DIR", cfg.artifacts_dir)) / "config.json"
        if path.exists():
            try:
                d = json.loads(path.read_text())
                for k, v in d.items():
                    if hasattr(cfg, k):
                        if k in ("train_seeds", "ks"):
                            v = tuple(v)
                        setattr(cfg, k, v)
            except Exception:
                pass
        # environment overrides (MVR_CLIP_MODEL=..., etc.)
        for k in list(asdict(cfg).keys()):
            env = os.environ.get("MVR_" + k.upper())
            if env is None:
                continue
            cur = getattr(cfg, k)
            if isinstance(cur, bool):
                setattr(cfg, k, env.lower() in ("1", "true", "yes"))
            elif isinstance(cur, int):
                setattr(cfg, k, int(env))
            elif isinstance(cur, float):
                setattr(cfg, k, float(env))
            elif isinstance(cur, tuple):
                setattr(cfg, k, tuple(int(x) for x in env.split(",")))
            else:
                setattr(cfg, k, env)
        return cfg


def method_label(m: str) -> str:
    return METHOD_INFO.get(m, (m, ""))[0]

#### `mvr/data.py`
**Data ingestion** – downloads annotations + videos from Hugging Face, merges caption sources, samples the stratified index, decodes TSN-style uniformly spaced frames with OpenCV, validates clips and saves thumbnails.

In [ ]:
%%writefile mvr/data.py
"""Data ingestion: MSR-VTT annotations, stratified sampling, video extraction
and frame decoding/pre-processing.

Data source: Hugging Face dataset repo ``friedrichor/MSR-VTT``
  * msrvtt_test_1k.json   - 1k-A test videos (category, 1 caption each)
  * msrvtt_train_9k.json  - 9k training videos (category, 20 captions each)
  * raw_data/MSRVTT_data.json - original annotation (all 20 captions / video)
  * raw_data/category.txt - category names
  * MSRVTT_Videos.zip     - all 10k .mp4 clips

Protocol implemented (as per the assignment):
  * Index  : 100 clips = 5 clips x 20 categories, drawn from the 1k-A TEST videos
  * Queries: 200 = 2 human captions per index clip
  * Train  : a disjoint pool from the 9k TRAIN videos, used only to fit the
             learned temporal heads (never overlaps with the index).
"""
from __future__ import annotations

import json
import random
import re
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Callable, Optional

import numpy as np

from .config import MSRVTT_CATEGORIES, Config

ProgressFn = Optional[Callable[[float, str], None]]


def _p(progress: ProgressFn, frac: float, msg: str) -> None:
    if progress:
        progress(min(max(frac, 0.0), 1.0), msg)
    else:
        print(f"[{frac*100:5.1f}%] {msg}", flush=True)


# ---------------------------------------------------------------------------
# Fetching files (Hugging Face hub or an offline local mirror)
# ---------------------------------------------------------------------------
def fetch_repo_file(cfg: Config, filename: str) -> Path:
    """Return a local path to ``filename`` from the dataset repo.

    Order: (1) already in data_dir, (2) local mirror dir, (3) HF hub download.
    """
    target = Path(cfg.data_dir) / filename
    if target.exists() and target.stat().st_size > 0:
        return target
    if cfg.local_repo_dir:
        src = Path(cfg.local_repo_dir) / filename
        if src.exists():
            return src
        raise FileNotFoundError(f"{filename} not found in local mirror {cfg.local_repo_dir}")
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(repo_id=cfg.hf_dataset_repo, filename=filename,
                           repo_type="dataset", local_dir=cfg.data_dir)
    return Path(path)


def _load_json_records(path: Path) -> list:
    """Load either a JSON array or JSON-lines file."""
    txt = path.read_text(encoding="utf-8").strip()
    if not txt:
        return []
    if txt[0] == "[":
        return json.loads(txt)
    if txt[0] == "{":
        try:
            obj = json.loads(txt)
            return obj if isinstance(obj, list) else [obj]
        except json.JSONDecodeError:
            pass
    return [json.loads(line) for line in txt.splitlines() if line.strip()]


def _vid_key(v) -> str:
    """Normalise video identifiers: 7020 / 'video7020' / 'video7020.mp4' -> 'video7020'."""
    s = str(v).strip()
    s = Path(s).stem if s.endswith(".mp4") else s
    if s.isdigit():
        s = f"video{s}"
    return s


def load_category_names(cfg: Config) -> list[str]:
    names = list(MSRVTT_CATEGORIES)
    try:
        p = fetch_repo_file(cfg, "raw_data/category.txt")
        for line in p.read_text(encoding="utf-8").splitlines():
            parts = re.split(r"[\t,]+|\s{2,}", line.strip())
            nums = [x for x in parts if x.strip().isdigit()]
            words = [x for x in parts if x.strip() and not x.strip().isdigit()]
            if nums and words and int(nums[0]) < 20:
                names[int(nums[0])] = words[0].strip()
    except Exception:
        pass  # fall back to canonical names
    return names


def load_annotations(cfg: Config, progress: ProgressFn = None) -> dict:
    """Merge all available annotation files into one table.

    Returns {video_id: {"category": int, "captions": [str,...], "split": str,
                        "url": str, "start": float, "end": float}}
    """
    table: dict[str, dict] = {}

    def upsert(vid, split=None, category=None, captions=None, url=None, start=None, end=None):
        rec = table.setdefault(vid, {"category": None, "captions": [], "split": None,
                                     "url": "", "start": None, "end": None})
        if category is not None and rec["category"] is None:
            rec["category"] = int(category)
        if split and rec["split"] in (None, "raw"):
            rec["split"] = split
        if url and not rec["url"]:
            rec["url"] = url
        if start is not None and rec["start"] is None:
            rec["start"], rec["end"] = start, end
        for c in captions or []:
            c = str(c).strip()
            if c and c not in rec["captions"]:
                rec["captions"].append(c)

    files = [("msrvtt_test_1k.json", "test1k"), ("msrvtt_train_9k.json", "train9k")]
    for i, (fname, split) in enumerate(files):
        _p(progress, 0.05 + 0.1 * i, f"Downloading annotation {fname}")
        path = fetch_repo_file(cfg, fname)
        for r in _load_json_records(path):
            vid = _vid_key(r.get("video_id") or r.get("video") or r.get("id"))
            caps = r.get("caption", r.get("captions", []))
            caps = [caps] if isinstance(caps, str) else list(caps or [])
            upsert(vid, split, r.get("category"), caps, r.get("url"),
                   r.get("start time"), r.get("end time"))

    # Original annotation: all 20 captions for every video (incl. test 1k-A).
    try:
        _p(progress, 0.30, "Downloading raw_data/MSRVTT_data.json (all 20 captions/video)")
        raw = json.loads(fetch_repo_file(cfg, "raw_data/MSRVTT_data.json").read_text(encoding="utf-8"))
        for v in raw.get("videos", []):
            upsert(_vid_key(v.get("video_id")), None, v.get("category"), None,
                   v.get("url"), v.get("start time"), v.get("end time"))
        for s in raw.get("sentences", []):
            vid = _vid_key(s.get("video_id"))
            if vid in table:
                upsert(vid, captions=[s.get("caption", "")])
    except Exception as e:  # not fatal - the sampler falls back to train9k videos
        _p(progress, 0.30, f"MSRVTT_data.json unavailable ({e}); continuing with split files only")
    return table


# ---------------------------------------------------------------------------
# Video archive handling
# ---------------------------------------------------------------------------
class VideoArchive:
    """Extracts individual clips from MSRVTT_Videos.zip (or a local folder)."""

    def __init__(self, cfg: Config, progress: ProgressFn = None):
        self.cfg = cfg
        self.out_dir = cfg.videos_dir
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.members: dict[str, str] = {}
        self.zf: zipfile.ZipFile | None = None
        self.local_folder: Path | None = None
        manual = Path(cfg.data_dir) / "videos_manual"   # user may drop .mp4s here
        if manual.exists() and any(manual.glob("*.mp4")):
            self.local_folder = manual
            return
        _p(progress, 0.35, "Fetching MSRVTT_Videos.zip (~2.2 GB, cached after first run)")
        zpath = fetch_repo_file(cfg, "MSRVTT_Videos.zip")
        self.zf = zipfile.ZipFile(zpath)
        for name in self.zf.namelist():
            if name.lower().endswith(".mp4") and not Path(name).name.startswith("._"):
                self.members[Path(name).stem] = name

    def available(self, vid: str) -> bool:
        if (self.out_dir / f"{vid}.mp4").exists():
            return True
        if self.local_folder is not None:
            return (self.local_folder / f"{vid}.mp4").exists()
        return vid in self.members

    def extract(self, vid: str) -> Path | None:
        dst = self.out_dir / f"{vid}.mp4"
        if dst.exists() and dst.stat().st_size > 0:
            return dst
        if self.local_folder is not None:
            src = self.local_folder / f"{vid}.mp4"
            if not src.exists():
                return None
            shutil.copy(src, dst)
            return dst
        name = self.members.get(vid)
        if name is None or self.zf is None:
            return None
        with self.zf.open(name) as fsrc, open(dst, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)
        return dst


# ---------------------------------------------------------------------------
# Frame decoding & pre-processing
# ---------------------------------------------------------------------------
def segment_indices(n_frames: int, num: int) -> np.ndarray:
    """TSN-style uniform sampling: mid-point of ``num`` equal temporal segments."""
    if n_frames <= 0:
        return np.zeros(num, dtype=int)
    idx = ((np.arange(num) + 0.5) * n_frames / num).astype(int)
    return np.clip(idx, 0, n_frames - 1)


def read_video_frames(path: str | Path, num: int) -> tuple[list[np.ndarray], dict]:
    """Decode ``num`` uniformly-spaced RGB frames with OpenCV.

    Frames are read sequentially (grab/retrieve) which is frame-accurate and
    robust to the unreliable seeking of some MP4 encodes. Returns (frames, meta).
    Raises ValueError for unreadable/corrupt videos.
    """
    import cv2

    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f"cannot open {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    frames: list[np.ndarray] = []
    if n > 0:
        wanted = segment_indices(n, num)
        wanted_set = defaultdict(int)
        for w in wanted:
            wanted_set[int(w)] += 1
        i = 0
        last_ok = None
        while i <= wanted.max():
            ok = cap.grab()
            if not ok:
                break
            if i in wanted_set:
                ok, f = cap.retrieve()
                if ok:
                    last_ok = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
                    frames.extend([last_ok] * wanted_set[i])
            i += 1
    cap.release()
    if len(frames) < num:
        # frame count metadata was wrong -> decode everything, then sample
        cap = cv2.VideoCapture(str(path))
        allf = []
        while True:
            ok, f = cap.read()
            if not ok:
                break
            allf.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        cap.release()
        if not allf:
            raise ValueError(f"no decodable frames in {path}")
        n = len(allf)
        frames = [allf[i] for i in segment_indices(n, num)]
    duration = n / fps if fps > 0 else 0.0
    return frames, {"fps": fps, "n_frames": n, "width": width, "height": height,
                    "duration": round(duration, 2)}


def is_browser_playable(path: Path) -> bool:
    import cv2
    cap = cv2.VideoCapture(str(path))
    fourcc = int(cap.get(cv2.CAP_PROP_FOURCC))
    cap.release()
    code = "".join(chr((fourcc >> 8 * i) & 0xFF) for i in range(4)).lower()
    return code in ("avc1", "h264", "x264", "vp80", "vp90", "av01")


def make_browser_playable(path: Path) -> Path:
    """Best-effort transcode to H.264 if the clip codec is not playable in a browser."""
    try:
        if is_browser_playable(path):
            return path
        ffmpeg = shutil.which("ffmpeg")
        if ffmpeg is None:
            try:
                import imageio_ffmpeg  # type: ignore
                ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
            except Exception:
                return path
        import subprocess
        tmp = path.with_suffix(".h264.mp4")
        subprocess.run([ffmpeg, "-y", "-loglevel", "error", "-i", str(path), "-c:v", "libx264",
                        "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-an", str(tmp)],
                       check=True, timeout=120)
        tmp.replace(path)
    except Exception:
        pass
    return path


def save_thumb_strip(frames: list[np.ndarray], out: Path, height: int = 96) -> None:
    """Save a horizontal strip of sampled frames (used by the dashboard)."""
    from PIL import Image
    ims = []
    for f in frames:
        im = Image.fromarray(f)
        w = int(im.width * height / max(im.height, 1))
        ims.append(im.resize((max(w, 1), height)))
    strip = Image.new("RGB", (sum(i.width for i in ims), height))
    x = 0
    for im in ims:
        strip.paste(im, (x, 0))
        x += im.width
    out.parent.mkdir(parents=True, exist_ok=True)
    strip.save(out, quality=85)


# ---------------------------------------------------------------------------
# Content filter: academic submission -> exclude clips described with sexual /
# explicit language (MSR-VTT is crowd-captioned web video and contains a few).
# ---------------------------------------------------------------------------
SENSITIVE_PATTERN = re.compile(
    r"\b(sexy|sexily|sexual\w*|sex|nude|nudity|naked|topless|lingerie|porn\w*|erotic\w*|seductive\w*|"
    r"seduc\w*|strip(per|pers|tease|teasing)|twerk\w*|boobs?|breasts?|cleavage|bikini\w*|"
    r"underwear|panties|bra|hot girls?|kiss(es|ing)?|making out)\b", re.IGNORECASE)


def is_sensitive(captions) -> bool:
    """True if ANY caption of the clip uses sexual/explicit wording."""
    return any(SENSITIVE_PATTERN.search(c or "") for c in captions)


# ---------------------------------------------------------------------------
# Subset construction (100 index clips / 200 queries / disjoint train pool)
# ---------------------------------------------------------------------------
def prepare_subset(cfg: Config, progress: ProgressFn = None, force: bool = False) -> dict:
    if cfg.subset_path.exists() and not force:
        subset = json.loads(cfg.subset_path.read_text())
        flagged = [v["video_id"] for v in subset["index"] if is_sensitive(v.get("all_captions", v["queries"]))]
        flagged += [t["video_id"] for t in subset.get("train", []) if is_sensitive(t["captions"])]
        if flagged:
            _p(progress, 0.01, f"Cached subset contains {len(flagged)} clip(s) excluded by the content filter "
                               f"({', '.join(flagged[:5])}) - rebuilding")
        elif all((cfg.videos_dir / f"{v['video_id']}.mp4").exists() for v in subset["index"]):
            _p(progress, 1.0, "Subset already prepared (cached)")
            return subset

    rng = random.Random(cfg.seed)
    cat_names = load_category_names(cfg)
    table = load_annotations(cfg, progress)
    archive = VideoArchive(cfg, progress)

    def usable(vid, rec, min_caps):
        return rec["category"] is not None and len(rec["captions"]) >= min_caps and archive.available(vid)

    # ---- index: prefer the official 1k-A test videos, then held-out train9k ----
    by_cat_test, by_cat_other = defaultdict(list), defaultdict(list)
    for vid, rec in sorted(table.items()):
        if not usable(vid, rec, cfg.captions_per_video):
            continue
        (by_cat_test if rec["split"] == "test1k" else by_cat_other)[rec["category"]].append(vid)

    index, used, excluded = [], set(), []
    n_cat = 20
    for ci, c in enumerate(range(n_cat)):
        _p(progress, 0.40 + 0.25 * ci / n_cat, f"Sampling + validating index clips for '{cat_names[c]}'")
        pool_a = by_cat_test[c][:]
        pool_b = by_cat_other[c][:]
        rng.shuffle(pool_a)
        rng.shuffle(pool_b)
        chosen = []
        for vid in pool_a + pool_b:
            if len(chosen) == cfg.videos_per_category:
                break
            if is_sensitive(table[vid]["captions"]):
                excluded.append(vid)
                continue
            path = archive.extract(vid)
            if path is None:
                continue
            try:
                frames, meta = read_video_frames(path, cfg.num_frames)   # validates decodability
            except Exception:
                continue
            make_browser_playable(path)
            save_thumb_strip(frames[:: max(1, len(frames) // 8)], cfg.thumbs_dir / f"{vid}.jpg")
            rec = table[vid]
            caps = rec["captions"]
            # caption 1 = the official 1k-A caption (first stored); caption 2 = random other one
            q = [caps[0]] + rng.sample(caps[1:], k=min(cfg.captions_per_video - 1, len(caps) - 1))
            chosen.append({"video_id": vid, "category": c, "category_name": cat_names[c],
                           "queries": q, "all_captions": caps, "url": rec["url"],
                           "source_split": rec["split"], **meta})
            used.add(vid)
        if len(chosen) < cfg.videos_per_category:
            raise RuntimeError(f"Only {len(chosen)} usable clips found for category {c} ({cat_names[c]})")
        index.extend(chosen)

    queries = []
    for vi, v in enumerate(index):
        for qi, q in enumerate(v["queries"]):
            queries.append({"qid": len(queries), "text": q, "video_idx": vi, "video_id": v["video_id"],
                            "category": v["category"]})

    # ---- disjoint training pool for learned temporal heads (train9k only) ----
    train = []
    n_train_per_cat = cfg.eff_train_per_cat()
    if n_train_per_cat > 0:
        tr_by_cat = defaultdict(list)
        for vid, rec in sorted(table.items()):
            if vid in used or rec["split"] != "train9k":
                continue
            if usable(vid, rec, 5):
                tr_by_cat[rec["category"]].append(vid)
        for ci, c in enumerate(range(n_cat)):
            _p(progress, 0.65 + 0.33 * ci / n_cat, f"Extracting training clips for '{cat_names[c]}'")
            pool = tr_by_cat[c][:]
            rng.shuffle(pool)
            got = 0
            for vid in pool:
                if got == n_train_per_cat:
                    break
                if is_sensitive(table[vid]["captions"]):
                    excluded.append(vid)
                    continue
                if archive.extract(vid) is None:
                    continue
                train.append({"video_id": vid, "category": c, "captions": table[vid]["captions"][:20]})
                got += 1

    assert not (set(v["video_id"] for v in index) & set(t["video_id"] for t in train)), "index/train overlap!"

    # caption lexicon for spelling correction - built ONLY from non-index videos (no leakage)
    from collections import Counter
    lex = Counter()
    for vid, rec in table.items():
        if vid in used:
            continue
        for c in rec["captions"]:
            lex.update(re.findall(r"[a-z]+", c.lower()))
    (Path(cfg.data_dir) / "lexicon.json").write_text(json.dumps(dict(lex.most_common())))
    subset = {"category_names": cat_names, "index": index, "queries": queries, "train": train,
              "protocol": {"videos_per_category": cfg.videos_per_category,
                           "captions_per_video": cfg.captions_per_video,
                           "n_index": len(index), "n_queries": len(queries), "n_train": len(train),
                           "seed": cfg.seed, "n_excluded_by_content_filter": len(set(excluded))}}
    cfg.subset_path.parent.mkdir(parents=True, exist_ok=True)
    cfg.subset_path.write_text(json.dumps(subset, indent=1))
    _p(progress, 1.0, f"Prepared {len(index)} index clips, {len(queries)} queries, {len(train)} training clips")
    return subset


def load_subset(cfg: Config) -> dict | None:
    if cfg.subset_path.exists():
        return json.loads(cfg.subset_path.read_text())
    return None

#### `mvr/encoders.py`
**Encoders** – CLIP pre-processing, frame and text embedding into the shared space (L2-normalised), and the optional X-CLIP scorer.

In [ ]:
%%writefile mvr/encoders.py
"""Pretrained encoders that map video frames and text into ONE shared space.

* CLIP (ViT image tower + Transformer text tower, contrastively pre-trained on
  400M image-text pairs). Frame embeddings and query embeddings are both
  projected by CLIP's own projection heads into a 512-d joint space and
  L2-normalised, so cosine similarity == dot product.
* X-CLIP (optional): a video-native model with cross-frame attention, used as an
  external spatio-temporal reference.

Only stable, version-independent sub-module calls are used
(``vision_model -> visual_projection`` and ``text_model -> text_projection``)
so the code works with transformers 4.3x as well as 5.x.
"""
from __future__ import annotations

from typing import Sequence

import numpy as np

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def preprocess_frames(frames: Sequence[np.ndarray], size: int = 224, mean=CLIP_MEAN, std=CLIP_STD):
    """CLIP pre-processing: bicubic resize of the short side -> centre crop ->
    scale to [0,1] -> per-channel normalisation. Returns float tensor (N,3,S,S)."""
    import torch
    from PIL import Image

    out = []
    for f in frames:
        im = Image.fromarray(f).convert("RGB")
        w, h = im.size
        s = size / min(w, h)
        im = im.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BICUBIC)
        w, h = im.size
        left, top = (w - size) // 2, (h - size) // 2
        im = im.crop((left, top, left + size, top + size))
        out.append(np.asarray(im, dtype=np.float32) / 255.0)
    x = torch.from_numpy(np.stack(out)).permute(0, 3, 1, 2)
    m = torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)
    s = torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1)
    return (x - m) / s


def _l2n(x: np.ndarray) -> np.ndarray:
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12, None)


class ClipEncoder:
    """Wraps a Hugging Face CLIP checkpoint for frame & text embedding."""

    def __init__(self, model_name: str, device: str = "cpu"):
        import torch
        from transformers import CLIPModel, CLIPTokenizer

        self.device = device
        self.dtype = torch.float16 if device.startswith("cuda") else torch.float32
        self.model = CLIPModel.from_pretrained(model_name).to(device=device, dtype=self.dtype).eval()
        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)
        self.image_size = int(self.model.config.vision_config.image_size)
        self.max_len = int(getattr(self.model.config.text_config, "max_position_embeddings", 77))
        self.dim = int(self.model.config.projection_dim)
        self.name = model_name

    # ----- images -----
    def encode_frames(self, frames: Sequence[np.ndarray], batch_size: int = 64) -> np.ndarray:
        import torch

        feats = []
        for i in range(0, len(frames), batch_size):
            px = preprocess_frames(frames[i:i + batch_size], self.image_size).to(self.device, self.dtype)
            with torch.no_grad():
                out = self.model.vision_model(pixel_values=px)
                emb = self.model.visual_projection(out.pooler_output)
            feats.append(emb.float().cpu().numpy())
        return _l2n(np.concatenate(feats, 0))

    # ----- text -----
    def tokenize(self, texts: Sequence[str]):
        return self.tokenizer(list(texts), padding=True, truncation=True,
                              max_length=self.max_len, return_tensors="pt")

    def encode_text(self, texts: Sequence[str], batch_size: int = 256) -> np.ndarray:
        import torch

        feats = []
        for i in range(0, len(texts), batch_size):
            tok = self.tokenize(texts[i:i + batch_size])
            with torch.no_grad():
                out = self.model.text_model(input_ids=tok["input_ids"].to(self.device),
                                            attention_mask=tok["attention_mask"].to(self.device))
                emb = self.model.text_projection(out.pooler_output)
            feats.append(emb.float().cpu().numpy())
        return _l2n(np.concatenate(feats, 0))

    def token_count(self, text: str) -> int:
        """Number of BPE tokens incl. <sot>/<eot> (no truncation)."""
        return len(self.tokenizer(text, truncation=False)["input_ids"])

    def word_pieces(self, word: str) -> list[str]:
        return self.tokenizer.tokenize(word)

    def in_vocab(self, word: str) -> bool:
        """True if the word is a single whole-word BPE token in CLIP's vocabulary."""
        vocab = self.tokenizer.get_vocab()
        return (word.lower() + "</w>") in vocab


class XClipScorer:
    """Optional X-CLIP video-text scorer (video-native spatio-temporal model).

    X-CLIP conditions its text embedding on the video (prompt generator), so a
    (query x video) score matrix is produced by a joint forward pass.
    """

    def __init__(self, model_name: str, device: str = "cpu"):
        import torch
        from transformers import CLIPTokenizer, XCLIPModel

        self.device = device
        self.model = XCLIPModel.from_pretrained(model_name).to(device).eval()
        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)
        self.num_frames = int(getattr(self.model.config.vision_config, "num_frames", 8))
        self.image_size = int(self.model.config.vision_config.image_size)
        self.torch = torch
        # use the normalisation the checkpoint was trained with (X-CLIP ships ImageNet mean/std)
        self.mean, self.std = CLIP_MEAN, CLIP_STD
        try:
            from transformers import AutoImageProcessor
            ip = AutoImageProcessor.from_pretrained(model_name)
            if getattr(ip, "image_mean", None) and getattr(ip, "image_std", None):
                self.mean, self.std = tuple(ip.image_mean), tuple(ip.image_std)
        except Exception:
            pass

    def video_pixels(self, frames: Sequence[np.ndarray]):
        idx = np.linspace(0, len(frames) - 1, self.num_frames).round().astype(int)
        return preprocess_frames([frames[i] for i in idx], self.image_size, self.mean, self.std)  # (T,3,H,W)

    def score_matrix(self, texts: Sequence[str], videos_pixels: list, batch_videos: int = 8) -> np.ndarray:
        torch = self.torch
        tok = self.tokenizer(list(texts), padding=True, truncation=True, max_length=77, return_tensors="pt")
        tok = {k: v.to(self.device) for k, v in tok.items()}
        cols = []
        for i in range(0, len(videos_pixels), batch_videos):
            px = torch.stack(videos_pixels[i:i + batch_videos]).to(self.device)  # (B,T,3,H,W)
            with torch.no_grad():
                out = self.model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"],
                                 pixel_values=px)
            cols.append(out.logits_per_text.float().cpu().numpy())  # (n_text, B)
        return np.concatenate(cols, axis=1)  # (n_text, n_videos)

#### `mvr/temporal.py`
**Temporal representations** – parameter-free aggregators and learned temporal heads (MLP control, Set Transformer, Temporal Transformer, LSTM) trained with symmetric InfoNCE.

In [ ]:
%%writefile mvr/temporal.py
"""Video-level representations built from per-frame CLIP embeddings.

Parameter-free (zero-shot) aggregators
    single_frame  : middle frame only
    mean_pool     : average of frame embeddings  (baseline in the research question)
    max_pool      : element-wise max
    query_attn    : query-conditioned softmax pooling of frame similarities

Learned heads (trained on a disjoint MSR-VTT training pool, CLIP frozen)
    mlp_adapter   : residual MLP on the mean-pooled vector  -> CONTROL (domain
                    adaptation without any inter-frame modelling)
    set_tfm       : Transformer over frames WITHOUT positional embeddings
                    (frame interactions, permutation-invariant)
    temporal_tfm  : Transformer over frames WITH temporal position embeddings
                    (explicit temporal order, CLIP4Clip "seqTransf")
    seq_lstm      : LSTM over ordered frames (CLIP4Clip "seqLSTM")

All learned heads are residual + zero-initialised, i.e. at initialisation they
are *exactly* mean pooling; any change in performance is therefore caused by
what the head learns on top of the baseline.
"""
from __future__ import annotations

import json
import math
import time
from pathlib import Path
from typing import Callable, Optional

import numpy as np

ProgressFn = Optional[Callable[[float, str], None]]


def l2n(x: np.ndarray) -> np.ndarray:
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), 1e-12, None)


# ---------------------------------------------------------------------------
# Parameter-free aggregation
# ---------------------------------------------------------------------------
def pool_video(frames: np.ndarray, method: str) -> np.ndarray:
    """frames: (N,T,D) L2-normalised -> (N,D) L2-normalised video vectors."""
    if method == "single_frame":
        return l2n(frames[:, frames.shape[1] // 2])
    if method == "mean_pool":
        return l2n(frames.mean(1))
    if method == "max_pool":
        return l2n(frames.max(1))
    raise ValueError(method)


def query_attn_scores(q: np.ndarray, frames: np.ndarray, tau: float) -> tuple[np.ndarray, np.ndarray]:
    """Query-conditioned temporal pooling.

    s_{q,n,t} = <q, f_{n,t}>;  w = softmax_t(s / tau);  S_{q,n} = sum_t w * s.
    tau -> inf  == mean pooling of similarities; tau -> 0 == max over frames.
    Returns (S (Q,N), weights (Q,N,T)).
    """
    s = np.einsum("qd,ntd->qnt", q, frames)
    z = s / tau
    z = z - z.max(-1, keepdims=True)
    w = np.exp(z)
    w = w / w.sum(-1, keepdims=True)
    return (w * s).sum(-1), w


# ---------------------------------------------------------------------------
# Learned temporal heads
# ---------------------------------------------------------------------------
def _torch():
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    return torch, nn, F


def build_head(arch: str, dim: int, max_frames: int = 16, layers: int = 2, heads: int = 8,
               dropout: float = 0.1):
    torch, nn, F = _torch()

    class TemporalHead(nn.Module):
        def __init__(self):
            super().__init__()
            self.arch = arch
            if arch == "mlp_adapter":
                self.f = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 2 * dim), nn.GELU(),
                                       nn.Dropout(dropout), nn.Linear(2 * dim, dim))
                last = self.f[-1]
            elif arch in ("set_tfm", "temporal_tfm"):
                self.pos = nn.Embedding(max_frames, dim) if arch == "temporal_tfm" else None
                if self.pos is not None:
                    nn.init.normal_(self.pos.weight, std=0.02)
                layer = nn.TransformerEncoderLayer(dim, heads, 4 * dim, dropout, batch_first=True,
                                                   norm_first=True, activation="gelu")
                self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
                self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim))
                last = self.out[-1]
            elif arch == "seq_lstm":
                self.lstm = nn.LSTM(dim, dim, batch_first=True)
                self.out = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim))
                last = self.out[-1]
            else:
                raise ValueError(arch)
            nn.init.zeros_(last.weight)   # residual zero-init => starts as mean pooling
            nn.init.zeros_(last.bias)

        def forward(self, x, pos_idx=None):  # x: (B,T,D)
            if self.arch == "mlp_adapter":
                base = x.mean(1)
                v = base + self.f(base)
            elif self.arch in ("set_tfm", "temporal_tfm"):
                h = x
                if self.pos is not None:
                    if pos_idx is None:
                        pos_idx = torch.arange(x.shape[1], device=x.device)
                    h = h + self.pos(pos_idx).unsqueeze(0) if pos_idx.dim() == 1 else h + self.pos(pos_idx)
                h = self.enc(h)
                v = (x + self.out(h)).mean(1)
            else:  # seq_lstm
                h, _ = self.lstm(x)
                v = (x + self.out(h)).mean(1)
            return F.normalize(v, dim=-1)

    return TemporalHead()


def head_embed(head, frames: np.ndarray, pos_idx: np.ndarray | None = None, device: str = "cpu") -> np.ndarray:
    torch, _, _ = _torch()
    head.eval()
    with torch.no_grad():
        x = torch.from_numpy(frames.astype(np.float32)).to(device)
        p = None if pos_idx is None else torch.from_numpy(np.asarray(pos_idx)).long().to(device)
        return head(x, p).cpu().numpy()


def train_head(arch: str, train_frames: np.ndarray, train_text: np.ndarray, text_mask: np.ndarray,
               seed: int, epochs: int = 40, lr: float = 1e-4, weight_decay: float = 0.05,
               batch: int = 128, layers: int = 2, heads: int = 8, device: str = "cpu",
               progress: ProgressFn = None) -> tuple[object, dict]:
    """Contrastive (symmetric InfoNCE) training of a temporal head on frozen CLIP features.

    train_frames (M,T,D), train_text (M,C,D) caption embeddings, text_mask (M,C) bool.
    10% of the training videos are held out for early stopping (the evaluation
    index is never used for any training or model-selection decision).
    """
    torch, nn, F = _torch()
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    M, T, D = train_frames.shape
    perm = rng.permutation(M)
    n_val = max(10, int(0.1 * M))
    val_idx, tr_idx = perm[:n_val], perm[n_val:]

    head = build_head(arch, D, max_frames=T, layers=layers, heads=heads).to(device)
    logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.03), device=device))
    opt = torch.optim.AdamW(list(head.parameters()) + [logit_scale], lr=lr, weight_decay=weight_decay)
    steps_per_epoch = max(1, math.ceil(len(tr_idx) / batch))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps_per_epoch,
                                                pct_start=0.1)
    Xf = torch.from_numpy(train_frames.astype(np.float32)).to(device)
    Xt = torch.from_numpy(train_text.astype(np.float32)).to(device)

    def val_mrr() -> float:
        v = head_embed(head, train_frames[val_idx], device=device)
        caps, owner = [], []
        for j, vi in enumerate(val_idx):
            for c in np.where(text_mask[vi])[0][:5]:
                caps.append(train_text[vi, c])
                owner.append(j)
        S = np.stack(caps) @ v.T
        gt = S[np.arange(len(owner)), owner]
        ranks = 1 + (S > gt[:, None]).sum(1)
        return float((1.0 / ranks).mean())

    hist = {"epoch": [], "loss": [], "val_mrr": []}
    best, best_state, best_ep = val_mrr(), {k: v.detach().clone() for k, v in head.state_dict().items()}, 0
    hist["epoch"].append(0); hist["loss"].append(None); hist["val_mrr"].append(best)
    t0 = time.time()
    for ep in range(1, epochs + 1):
        head.train()
        order = rng.permutation(tr_idx)
        losses = []
        for b in range(0, len(order), batch):
            ids = order[b:b + batch]
            if len(ids) < 2:
                continue
            # one random caption per video
            cap = np.array([rng.choice(np.where(text_mask[i])[0]) for i in ids])
            t_emb = Xt[torch.as_tensor(ids, device=device), torch.as_tensor(cap, device=device)]
            x = Xf[torch.as_tensor(ids, device=device)]
            pos = None
            if rng.random() < 0.5:  # temporal sub-sampling augmentation (k frames, keep order)
                k = int(rng.choice([4, 8]))
                sel = np.sort(rng.choice(T, size=k, replace=False))
                x = x[:, torch.as_tensor(sel, device=device)]
                pos = torch.as_tensor(sel, device=device)
            v = head(x, pos)
            logits = logit_scale.exp().clamp(max=100) * t_emb @ v.t()
            lbl = torch.arange(len(ids), device=device)
            loss = 0.5 * (F.cross_entropy(logits, lbl) + F.cross_entropy(logits.t(), lbl))
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            losses.append(loss.item())
        m = val_mrr()
        hist["epoch"].append(ep); hist["loss"].append(float(np.mean(losses)) if losses else None)
        hist["val_mrr"].append(m)
        if m > best:
            best, best_ep = m, ep
            best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
        if progress:
            progress(ep / epochs, f"{arch} seed {seed}: epoch {ep}/{epochs} loss={hist['loss'][-1]:.3f} val MRR={m:.3f}")
    head.load_state_dict(best_state)
    hist.update({"best_val_mrr": best, "best_epoch": best_ep, "train_seconds": round(time.time() - t0, 1),
                 "n_train_videos": int(len(tr_idx)), "n_val_videos": int(len(val_idx)),
                 "n_params": int(sum(p.numel() for p in head.parameters()))})
    return head, hist


def save_head(head, hist: dict, path: Path) -> None:
    torch, _, _ = _torch()
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(head.state_dict(), path)
    path.with_suffix(".json").write_text(json.dumps(hist, indent=1))


def load_head(arch: str, dim: int, path: Path, max_frames: int, layers: int, heads: int, device: str = "cpu"):
    torch, _, _ = _torch()
    head = build_head(arch, dim, max_frames=max_frames, layers=layers, heads=heads)
    head.load_state_dict(torch.load(path, map_location=device))
    return head.to(device).eval()


def select_tau(train_frames: np.ndarray, train_text: np.ndarray, text_mask: np.ndarray,
               grid=(0.005, 0.01, 0.02, 0.05, 0.1, 0.2)) -> tuple[float, dict]:
    """Pick the query-attention temperature on the TRAINING pool (no test leakage)."""
    caps, owner = [], []
    for vi in range(len(train_frames)):
        c = np.where(text_mask[vi])[0]
        if len(c):
            caps.append(train_text[vi, c[0]])
            owner.append(vi)
    Q = np.stack(caps)
    res = {}
    for tau in grid:
        S, _ = query_attn_scores(Q, train_frames, tau)
        gt = S[np.arange(len(owner)), owner]
        ranks = 1 + (S > gt[:, None]).sum(1)
        res[tau] = float((1 / ranks).mean())
    best = max(res, key=res.get)
    return float(best), {str(k): v for k, v in res.items()}

#### `mvr/index.py`
**Vector index** – one FAISS `IndexFlatIP` per representation + the frame tensor for query-conditioned pooling.

In [ ]:
%%writefile mvr/index.py
"""Vector index over the 100 video clips.

For every video representation (method) we store an (N,D) matrix of
L2-normalised vectors in a FAISS ``IndexFlatIP`` (exact inner-product search;
inner product of unit vectors == cosine similarity). Frame-level vectors
(N*T,D) are also kept for query-conditioned pooling and per-frame
visualisation. Falls back to NumPy matrix products if FAISS is unavailable.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np

try:
    import faiss  # type: ignore
except Exception:  # pragma: no cover
    faiss = None


class VideoIndex:
    def __init__(self, video_ids: list[str], frames: np.ndarray, pooled: dict[str, np.ndarray],
                 tau: float = 0.01):
        self.video_ids = list(video_ids)
        self.frames = frames.astype(np.float32)          # (N,T,D)
        self.pooled = {k: v.astype(np.float32) for k, v in pooled.items()}
        self.tau = tau
        self.backend = "faiss.IndexFlatIP" if faiss is not None else "numpy"
        self._faiss = {}
        if faiss is not None:
            for m, M in self.pooled.items():
                ix = faiss.IndexFlatIP(M.shape[1])
                ix.add(np.ascontiguousarray(M))
                self._faiss[m] = ix

    @property
    def methods(self) -> list[str]:
        return list(self.pooled.keys()) + ["query_attn"]

    def scores(self, q: np.ndarray, method: str) -> np.ndarray:
        """Full similarity matrix (Q,N) for a batch of query embeddings."""
        q = np.atleast_2d(q).astype(np.float32)
        if method == "query_attn":
            from .temporal import query_attn_scores
            return query_attn_scores(q, self.frames, self.tau)[0]
        return q @ self.pooled[method].T

    def search(self, q: np.ndarray, method: str, k: int = 5) -> dict:
        """Top-K search for one query embedding. Returns ids, scores, per-frame sims."""
        t0 = time.perf_counter()
        q = np.atleast_2d(q).astype(np.float32)
        k = int(min(max(k, 1), len(self.video_ids)))
        if method == "query_attn":
            S = self.scores(q, method)[0]
            top = np.argsort(-S, kind="stable")[:k]
            sc = S[top]
        elif method in self._faiss:
            sc, top = self._faiss[method].search(q, k)
            sc, top = sc[0], top[0]
        else:
            S = self.scores(q, method)[0]
            top = np.argsort(-S, kind="stable")[:k]
            sc = S[top]
        latency = (time.perf_counter() - t0) * 1000
        frame_sims = np.einsum("d,ktd->kt", q[0], self.frames[top])
        return {"idx": top.tolist(), "scores": [float(s) for s in sc], "latency_ms": latency,
                "frame_sims": frame_sims}

    # ------------------------------------------------------------------
    def save(self, d: Path) -> None:
        d.mkdir(parents=True, exist_ok=True)
        np.save(d / "frames.npy", self.frames)
        for m, M in self.pooled.items():
            np.save(d / f"pooled_{m}.npy", M)
            if m in self._faiss:
                faiss.write_index(self._faiss[m], str(d / f"{m}.faiss"))
        (d / "meta.json").write_text(json.dumps({"video_ids": self.video_ids, "tau": self.tau,
                                                 "methods": list(self.pooled), "backend": self.backend}))

    @classmethod
    def load(cls, d: Path) -> "VideoIndex":
        meta = json.loads((d / "meta.json").read_text())
        frames = np.load(d / "frames.npy")
        pooled = {m: np.load(d / f"pooled_{m}.npy") for m in meta["methods"]}
        return cls(meta["video_ids"], frames, pooled, meta.get("tau", 0.01))

#### `mvr/query.py`
**Query processing / edge cases** – normalisation, unknown-word detection, frequency-aware spelling correction, prompt templates for short queries, chunking for long queries.

In [ ]:
%%writefile mvr/query.py
"""Natural-language query processing with explicit edge-case handling.

Edge cases handled
  * empty / whitespace / punctuation-only input  -> rejected with a clear message
  * unicode noise, control chars, repeated spaces -> NFKC normalisation + cleanup
  * unknown / misspelled vocabulary               -> detected against CLIP's BPE
        vocabulary and an MSR-VTT caption lexicon; optional edit-distance
        spelling correction (Norvig-style, edits <= 2, frequency-ranked)
  * very short queries (<= 2 content words)       -> prompt-template ensembling
        ("a video of {q}.", ...) which moves the query into CLIP's caption
        distribution
  * very long / detailed queries (> 77 BPE tokens)-> sentence-aware chunking;
        chunk embeddings are averaged instead of silently truncating
  * non-English input                             -> warning (CLIP text tower is English)
"""
from __future__ import annotations

import re
import string
import unicodedata
from collections import Counter
from dataclasses import dataclass, field

import numpy as np

STOPWORDS = set("""a an the of in on at to for with and or is are was were be being been by from
this that these those it its as into onto over under up down out off some any very
his her their our your my he she they we you i me him them who which while""".split())

TEMPLATES = ["a video of {}.", "a video showing {}.", "{}"]
MAX_CHARS = 1000


class QueryError(ValueError):
    pass


@dataclass
class ProcessedQuery:
    original: str
    normalized: str
    final_text: str
    texts_to_encode: list
    mode: str = "normal"                 # short | normal | long
    n_tokens: int = 0
    truncated_without_chunking: bool = False
    unknown_words: list = field(default_factory=list)
    corrections: dict = field(default_factory=dict)
    steps: list = field(default_factory=list)
    warnings: list = field(default_factory=list)

    def as_dict(self) -> dict:
        return self.__dict__.copy()


class SpellCorrector:
    """Frequency-ranked edit-distance corrector over a caption lexicon."""

    letters = string.ascii_lowercase

    def __init__(self, counts: Counter):
        self.counts = counts

    def _edits1(self, w):
        splits = [(w[:i], w[i:]) for i in range(len(w) + 1)]
        deletes = [a + b[1:] for a, b in splits if b]
        transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
        replaces = [a + c + b[1:] for a, b in splits if b for c in self.letters]
        inserts = [a + c + b for a, b in splits for c in self.letters]
        return set(deletes + transposes + replaces + inserts)

    def correct(self, w: str, in_clip_vocab: bool = False) -> str | None:
        """Return a correction for ``w`` or None.

        Crowd-sourced MSR-VTT captions themselves contain typos (e.g. 'playng'), so
        presence in the lexicon is not proof of correctness: a word is corrected when an
        edit-distance neighbour is MUCH more frequent than the word itself.
        """
        own = self.counts.get(w, 0)
        if own >= 1000:                       # clearly a real, common word
            return None
        cands = {e for e in self._edits1(w) if e in self.counts and e != w}
        if not cands and own == 0 and len(w) >= 5:
            cands = {e2 for e1 in self._edits1(w) for e2 in self._edits1(e1) if e2 in self.counts and e2 != w}
        if not cands:
            return None
        best = max(cands, key=self.counts.get)
        f = self.counts[best]
        if own > 0:
            need = max(20, 50 * own)          # rare variant of a very frequent word -> typo
        elif in_clip_vocab:
            need = 500                        # unseen in captions but a real BPE word: be conservative
        else:
            need = 5
        return best if f >= need else None


def build_lexicon(captions) -> Counter:
    c = Counter()
    for cap in captions:
        c.update(re.findall(r"[a-z]+", cap.lower()))
    return c


class QueryProcessor:
    def __init__(self, encoder, lexicon: Counter | None = None):
        self.enc = encoder
        self.lexicon = lexicon or Counter()
        self.speller = SpellCorrector(self.lexicon) if self.lexicon else None
        self.max_tokens = getattr(encoder, "max_len", 77)

    # ------------------------------------------------------------------
    @staticmethod
    def normalize(raw: str) -> str:
        if raw is None:
            raise QueryError("Please enter a query.")
        s = unicodedata.normalize("NFKC", str(raw))
        s = "".join(ch if unicodedata.category(ch)[0] != "C" else " " for ch in s)
        s = re.sub(r"https?://\S+", " ", s)
        s = re.sub(r"\s+", " ", s).strip().strip("\"'`")
        return s[:MAX_CHARS]

    def _in_clip_vocab(self, w: str) -> bool:
        try:
            return bool(self.enc.in_vocab(w))
        except Exception:
            return True

    def _chunks(self, text: str) -> list[str]:
        """Greedy sentence/clause packing so each chunk fits CLIP's 77-token window."""
        parts = [p.strip() for p in re.split(r"(?<=[.!?;])\s+|,\s+", text) if p.strip()]
        chunks, cur = [], ""
        for p in parts:
            cand = (cur + ", " + p) if cur else p
            if self.enc.token_count(cand) <= self.max_tokens:
                cur = cand
                continue
            if cur:
                chunks.append(cur)
            # a single clause longer than the window: split by words
            if self.enc.token_count(p) > self.max_tokens:
                words, buf = p.split(), ""
                for w in words:
                    c2 = (buf + " " + w).strip()
                    if self.enc.token_count(c2) > self.max_tokens:
                        chunks.append(buf)
                        buf = w
                    else:
                        buf = c2
                cur = buf
            else:
                cur = p
        if cur:
            chunks.append(cur)
        return chunks

    # ------------------------------------------------------------------
    def process(self, raw: str, auto_correct: bool = True, use_templates: bool = True,
                use_chunking: bool = True) -> ProcessedQuery:
        norm = self.normalize(raw)
        if not norm or not re.search(r"[A-Za-z0-9]", norm):
            raise QueryError("The query is empty or contains no letters/digits. "
                             "Please describe the video, e.g. 'a man is playing guitar on stage'.")
        pq = ProcessedQuery(original=str(raw), normalized=norm, final_text=norm, texts_to_encode=[norm])
        if norm != str(raw):
            pq.steps.append("normalised unicode / whitespace")

        non_ascii = sum(1 for ch in norm if ch.isalpha() and ord(ch) > 127)
        alpha = sum(1 for ch in norm if ch.isalpha()) or 1
        if non_ascii / alpha > 0.3:
            pq.warnings.append("Query looks non-English; CLIP's text encoder is trained on English captions, "
                               "so results may be unreliable.")

        # ---- unknown vocabulary / spelling ----
        words = re.findall(r"[A-Za-z]+", norm)
        text = norm
        for w in words:
            lw = w.lower()
            if len(lw) < 3 or lw.isdigit():
                continue
            in_vocab = self._in_clip_vocab(lw)
            fix = self.speller.correct(lw, in_vocab) if self.speller else None
            if fix:
                pq.unknown_words.append(w)
                if auto_correct:
                    pq.corrections[w] = fix
                    text = re.sub(rf"\b{re.escape(w)}\b", fix, text)
            elif lw not in self.lexicon and not in_vocab:
                pq.unknown_words.append(w)
        if pq.unknown_words:
            if pq.corrections:
                pq.steps.append("spelling corrected: " + ", ".join(f"{k}->{v}" for k, v in pq.corrections.items()))
            rest = [w for w in pq.unknown_words if w not in pq.corrections]
            if rest:
                pq.warnings.append("Out-of-vocabulary words (handled as BPE sub-words, may carry little meaning): "
                                   + ", ".join(rest))
        pq.final_text = text

        # ---- length handling ----
        content = [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]
        pq.n_tokens = self.enc.token_count(text)
        if pq.n_tokens > self.max_tokens:
            pq.mode = "long"
            if use_chunking:
                pq.texts_to_encode = self._chunks(text)
                pq.steps.append(f"long query ({pq.n_tokens} tokens > {self.max_tokens}): split into "
                                f"{len(pq.texts_to_encode)} chunks, embeddings averaged")
            else:
                pq.truncated_without_chunking = True
                pq.texts_to_encode = [text]
                pq.warnings.append(f"Query truncated to {self.max_tokens} tokens.")
        elif len(content) <= 2:
            pq.mode = "short"
            if use_templates:
                pq.texts_to_encode = [t.format(text.rstrip(".")) for t in TEMPLATES]
                pq.steps.append("short query: prompt-template ensemble " + str(TEMPLATES))
            else:
                pq.texts_to_encode = [text]
            if len(content) == 0:
                pq.warnings.append("Query has only stop-words; results will be weak.")
        else:
            pq.texts_to_encode = [text]
        return pq

    def encode(self, pq: ProcessedQuery) -> np.ndarray:
        e = self.enc.encode_text(pq.texts_to_encode).mean(0)
        return e / max(np.linalg.norm(e), 1e-12)

    def encode_many(self, texts, **kw) -> np.ndarray:
        """Batch helper used by the robustness experiments."""
        pqs = [self.process(t, **kw) for t in texts]
        flat, owner = [], []
        for i, pq in enumerate(pqs):
            flat.extend(pq.texts_to_encode)
            owner.extend([i] * len(pq.texts_to_encode))
        E = self.enc.encode_text(flat)
        out = np.zeros((len(pqs), E.shape[1]), dtype=np.float32)
        np.add.at(out, np.array(owner), E)
        return out / np.clip(np.linalg.norm(out, axis=1, keepdims=True), 1e-12, None)


# ---------------------------------------------------------------------------
# Query perturbations for the robustness study
# ---------------------------------------------------------------------------
def inject_typos(text: str, rng: np.random.Generator, rate: float = 0.35) -> str:
    words = text.split()
    out = []
    for w in words:
        if len(w) >= 4 and w.isalpha() and rng.random() < rate:
            i = int(rng.integers(1, len(w) - 1))
            op = rng.integers(0, 3)
            if op == 0:
                w = w[:i] + w[i + 1] + w[i] + w[i + 2:]          # transpose
            elif op == 1:
                w = w[:i] + w[i + 1:]                           # delete
            else:
                w = w[:i] + "abcdefghijklmnopqrstuvwxyz"[rng.integers(0, 26)] + w[i + 1:]  # substitute
        out.append(w)
    return " ".join(out)


def shorten(text: str, n_content: int = 2) -> str:
    """Keep only the first n content words (e.g. 'man guitar')."""
    content = [w for w in re.findall(r"[A-Za-z]+", text) if w.lower() not in STOPWORDS]
    return " ".join(content[:n_content]) or text

#### `mvr/metrics.py`
**Metrics & statistics** – standard IR metrics for both directions and paired significance tests.

In [ ]:
%%writefile mvr/metrics.py
"""Standard IR evaluation metrics and statistical tests.

Text->Video (T2V): each query has exactly ONE relevant video, so
    AP(q) = 1/rank(q)  =>  mAP == MRR   (reported for completeness)
Video->Text (V2T): each video has TWO relevant captions -> mAP differs from MRR.
"""
from __future__ import annotations

import numpy as np


def t2v_ranks(S: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """S (Q,N) similarity, gt (Q,) index of the relevant video -> 1-based ranks."""
    g = S[np.arange(len(gt)), gt]
    return 1 + (S > g[:, None]).sum(1)


def summarize_ranks(ranks: np.ndarray, ks=(1, 5, 10)) -> dict:
    ranks = np.asarray(ranks, dtype=float)
    out = {f"R@{k}": float((ranks <= k).mean() * 100) for k in ks}
    rr = 1.0 / ranks
    out["MRR"] = float(rr.mean())
    out["mAP"] = float(rr.mean())  # single relevant item
    out["nDCG@10"] = float(np.where(ranks <= 10, 1 / np.log2(ranks + 1), 0).mean())
    out["MdR"] = float(np.median(ranks))
    out["MnR"] = float(ranks.mean())
    return out


def v2t_metrics(S_vt: np.ndarray, rel: list[set], ks=(1, 5, 10)) -> tuple[dict, dict]:
    """S_vt (N,Q): video->caption similarities, rel[n] = set of relevant caption ids."""
    order = np.argsort(-S_vt, axis=1, kind="stable")
    first, aps, ndcgs = [], [], []
    for n in range(S_vt.shape[0]):
        hits = np.isin(order[n], list(rel[n]))
        pos = np.where(hits)[0] + 1
        first.append(pos[0])
        prec = np.arange(1, len(pos) + 1) / pos
        aps.append(prec.mean())
        dcg = sum(1 / np.log2(p + 1) for p in pos if p <= 10)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel[n]), 10)))
        ndcgs.append(dcg / idcg)
    first = np.array(first, dtype=float)
    out = {f"R@{k}": float((first <= k).mean() * 100) for k in ks}
    out["MRR"] = float((1 / first).mean())
    out["mAP"] = float(np.mean(aps))
    out["nDCG@10"] = float(np.mean(ndcgs))
    out["MdR"] = float(np.median(first))
    out["MnR"] = float(first.mean())
    return out, {"first_rank": first.tolist(), "ap": [float(a) for a in aps]}


# ---------------------------------------------------------------------------
# Significance testing (paired, query-level)
# ---------------------------------------------------------------------------
def paired_bootstrap(a: np.ndarray, b: np.ndarray, n: int = 10000, seed: int = 0) -> dict:
    """Bootstrap the mean difference (b - a) over queries. Returns delta, 95% CI, p."""
    rng = np.random.default_rng(seed)
    d = np.asarray(b, float) - np.asarray(a, float)
    idx = rng.integers(0, len(d), size=(n, len(d)))
    boots = d[idx].mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    # two-sided p-value: how often the bootstrap distribution crosses zero
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"delta": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi), "p_boot": float(min(1.0, p))}


def wilcoxon_p(a: np.ndarray, b: np.ndarray) -> float:
    try:
        from scipy.stats import wilcoxon
        d = np.asarray(b, float) - np.asarray(a, float)
        if np.allclose(d, 0):
            return 1.0
        return float(wilcoxon(a, b, zero_method="zsplit").pvalue)
    except Exception:
        return float("nan")


def holm(pvals: dict) -> dict:
    """Holm-Bonferroni adjustment of a {name: p} dict."""
    items = sorted([(p, k) for k, p in pvals.items() if p == p], key=lambda x: x[0])
    m = len(items)
    adj, running = {}, 0.0
    for i, (p, k) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        adj[k] = running
    for k, p in pvals.items():
        adj.setdefault(k, float("nan"))
    return adj

#### `mvr/control.py`
**Positive control** – synthetic task in which only frame order separates clips.

In [ ]:
%%writefile mvr/control.py
"""Positive-control experiment for the research question.

A synthetic retrieval task in which ONLY frame order distinguishes videos
("state A then B" vs "state B then A"). It verifies that the evaluation
set-up is able to detect a temporal benefit when one exists: order-aware heads
should succeed while mean pooling and order-free heads cannot. If, on real
MSR-VTT, the same heads show no gain, the absence is a property of the data,
not a failure of the temporal models.
"""
from __future__ import annotations

import numpy as np

from .temporal import head_embed, l2n, pool_video, train_head


def make_order_task(n_states=12, D=64, T=16, reps=6, seed=0):
    rng = np.random.default_rng(seed)
    S = l2n(rng.normal(size=(n_states, D)))
    W1, W2 = rng.normal(size=(D, D)) / 8, rng.normal(size=(D, D)) / 8
    frames, texts = [], []
    for a in range(n_states):
        for b in range(n_states):
            if a == b:
                continue
            for _ in range(reps):
                f = np.concatenate([np.repeat(S[a][None], T // 2, 0), np.repeat(S[b][None], T // 2, 0)])
                frames.append(l2n(f + 0.05 * rng.normal(size=f.shape)))
                texts.append(l2n(S[a] + S[b] + S[a] @ W1 + S[b] @ W2 + 0.05 * rng.normal(size=(5, D))))
    return np.array(frames, np.float32), np.array(texts, np.float32), reps


def _mrr(V, Q):
    S = Q @ V.T
    g = np.diag(S)
    return float((1 / (1 + (S > g[:, None]).sum(1))).mean())


def run_positive_control(epochs: int = 25, seed: int = 0) -> dict:
    F, Tx, reps = make_order_task(seed=seed)
    mask = np.ones(Tx.shape[:2], bool)
    te = np.arange(0, len(F), reps)
    tr = np.setdiff1d(np.arange(len(F)), te)
    Fte, Qte = F[te], Tx[te][:, 0]
    out = {"mean_pool": _mrr(pool_video(Fte, "mean_pool"), Qte),
           "max_pool": _mrr(pool_video(Fte, "max_pool"), Qte)}
    for arch in ("mlp_adapter", "set_tfm", "temporal_tfm", "seq_lstm"):
        h, _ = train_head(arch, F[tr], Tx[tr], mask[tr], seed=seed, epochs=epochs, lr=1e-3, heads=4, batch=128)
        out[arch] = _mrr(head_embed(h, Fte), Qte)
    return {"mrr": out, "n_test_queries": int(len(te)), "n_train_videos": int(len(tr))}

#### `mvr/analysis.py`
**Conclusions** – turns the measured numbers into the answer to Research Investigation 1.

In [ ]:
%%writefile mvr/analysis.py
"""Data-driven answer to Research Investigation 1:
"Is mean-pooling frame embeddings sufficient, or does explicitly modelling
temporal relationships give superior retrieval performance?"

All statements below are generated from the measured numbers (no hard-coded
claims), so the report always matches the run that produced it.
"""
from __future__ import annotations

from .config import method_label


def _sig(R, a, b):
    for row in R.get("significance", []):
        if row["method"] == a and row["reference"] == b:
            return row
    return None


def _fmt_delta(row, key="MRR", scale=1.0, unit=""):
    d = row[key]
    return (f"{d['delta']*scale:+.3f}{unit} (95% CI {d['ci_low']*scale:+.3f} to {d['ci_high']*scale:+.3f}; "
            f"Wilcoxon p={row['wilcoxon_p']:.3g}, Holm-adj. p={row['wilcoxon_p_holm']:.3g})")


def _is_sig(row, alpha=0.05):
    if row is None:
        return False
    ci = row["MRR"]
    excl0 = ci["ci_low"] > 0 or ci["ci_high"] < 0
    p = row.get("wilcoxon_p_holm", 1.0)
    return excl0 and (p == p) and p < alpha


def research_conclusions(R: dict) -> dict:
    t2v = R["t2v"]
    mp = t2v["mean_pool"]
    pts = []
    learned_present = all(m in t2v for m in ("mlp_adapter", "set_tfm", "temporal_tfm"))

    # 1. Baseline position
    pf = [m for m in ("single_frame", "mean_pool", "max_pool", "query_attn") if m in t2v]
    best_pf = max(pf, key=lambda m: t2v[m]["MRR"])
    sf = t2v.get("single_frame")
    if sf:
        d = mp["MRR"] - sf["MRR"]
        if d > 0.005:
            tail = "so aggregating evidence over the whole clip clearly helps."
        elif d < -0.005:
            tail = "so averaging all frames does not help over one representative frame in this run."
        else:
            tail = "so for these clips one representative frame carries almost all of the retrievable signal."
        pts.append(f"Mean pooling reaches R@1={mp['R@1']:.1f}% / MRR={mp['MRR']:.3f} vs. a single middle frame at "
                   f"R@1={sf['R@1']:.1f}% / MRR={sf['MRR']:.3f} ({d:+.3f} MRR), " + tail)
    if best_pf != "mean_pool":
        row = _sig(R, best_pf, "mean_pool")
        pts.append(f"Among parameter-free (zero-shot) aggregators the best is {method_label(best_pf)} "
                   f"(MRR {t2v[best_pf]['MRR']:.3f}); difference vs. mean pooling: "
                   + (_fmt_delta(row) if row else "n/a") + ".")
    else:
        pts.append("Among parameter-free (zero-shot) aggregators, mean pooling is the strongest; max pooling and "
                   "query-attentive pooling do not improve on it.")

    # 2. Frame-count saturation
    sw = R.get("frame_sweep", {}).get("mean_pool")
    if sw:
        ks = sorted(sw, key=int)
        best = max(sw[k]["MRR"] for k in ks)
        sat = next(k for k in ks if sw[k]["MRR"] >= 0.98 * best)
        pts.append(f"Mean pooling saturates quickly: with {sat} frame(s) it already achieves >=98% of its best MRR "
                   f"({sw[sat]['MRR']:.3f} vs {best:.3f}); adding more frames brings diminishing returns.")

    verdict = "inconclusive"
    if learned_present:
        ad, st, tt = t2v["mlp_adapter"], t2v["set_tfm"], t2v["temporal_tfm"]
        r_tt_mp = _sig(R, "temporal_tfm", "mean_pool")
        r_order = _sig(R, "temporal_tfm", "set_tfm")
        r_tt_ad = _sig(R, "temporal_tfm", "mlp_adapter")
        pts.append(f"Temporal Transformer (+positions) vs mean pooling: MRR {tt['MRR']:.3f} vs {mp['MRR']:.3f}, "
                   f"R@1 {tt['R@1']:.1f}% vs {mp['R@1']:.1f}%; delta MRR = " + (_fmt_delta(r_tt_mp) if r_tt_mp else "n/a") + ".")
        pts.append(f"Decomposition of that difference: (i) domain adaptation alone (MLP adapter, no temporal mixing) "
                   f"changes MRR by {ad['MRR']-mp['MRR']:+.3f}; (ii) adding cross-frame self-attention without order "
                   f"(Set Transformer) adds {st['MRR']-ad['MRR']:+.3f}; (iii) adding temporal ORDER (position "
                   f"embeddings) adds {tt['MRR']-st['MRR']:+.3f}"
                   + (f" [order effect: {_fmt_delta(r_order)}]" if r_order else "") + ".")
        if "seq_lstm" in t2v:
            ls = t2v["seq_lstm"]
            pts.append(f"The order-aware LSTM obtains MRR {ls['MRR']:.3f} (R@1 {ls['R@1']:.1f}%), i.e. "
                       f"{ls['MRR']-mp['MRR']:+.3f} vs mean pooling and {ls['MRR']-ad['MRR']:+.3f} vs the MLP control.")

        oa = R.get("order_ablation", {})
        if "temporal_tfm" in oa:
            o = oa["temporal_tfm"]
            pts.append(f"Shuffling the frame order at test time changes the Temporal Transformer's MRR from "
                       f"{o['original']['MRR']:.3f} to {o['shuffled']['MRR']:.3f} (reversed: {o['reversed']['MRR']:.3f}); "
                       f"order-agnostic methods are unaffected by construction (mean pooling: "
                       f"{oa['mean_pool']['original']['MRR']:.3f} -> {oa['mean_pool']['shuffled']['MRR']:.3f}). "
                       + ("The small change shows the model barely relies on frame order for these queries."
                          if abs(o['original']['MRR'] - o['shuffled']['MRR']) < 0.01 else
                          "The model does use frame order, but see the significance tests for whether this helps."))

        tl = R.get("temporal_language", {})
        if tl.get("n_temporal", 0) >= 5:
            bm = tl["by_method"]
            d_t = bm["temporal_tfm"]["temporal_MRR"] - bm["mean_pool"]["temporal_MRR"]
            d_o = bm["temporal_tfm"]["other_MRR"] - bm["mean_pool"]["other_MRR"]
            pts.append(f"On the {tl['n_temporal']} queries with explicit temporal language (e.g. 'then', 'after', "
                       f"'starts') the Temporal Transformer changes MRR by {d_t:+.3f} vs mean pooling, against "
                       f"{d_o:+.3f} on the remaining {tl['n_other']} queries.")

        temporal_beats_control = _is_sig(r_tt_ad, 0.05) and r_tt_ad["MRR"]["delta"] > 0
        order_helps = _is_sig(r_order, 0.05) and r_order["MRR"]["delta"] > 0
        tt_beats_mp = _is_sig(r_tt_mp, 0.05) and r_tt_mp["MRR"]["delta"] > 0
        if order_helps and temporal_beats_control:
            verdict = "temporal_better"
            answer = ("Explicit temporal modelling gives a statistically significant improvement over mean pooling "
                      "that is NOT explained by domain adaptation alone (it also beats the MLP control and the "
                      "order-free Set Transformer). Mean pooling is therefore not fully sufficient.")
        elif tt_beats_mp and not order_helps:
            verdict = "adaptation_not_order"
            answer = ("The trained temporal models beat zero-shot mean pooling, but the gain is explained by "
                      "learning on in-domain MSR-VTT data (the MLP control and the order-free Set Transformer "
                      "capture it) rather than by modelling temporal ORDER. Mean-pooled CLIP frame embeddings "
                      "are a sufficient representation of temporal content for this retrieval task; explicit "
                      "temporal modelling adds no significant benefit.")
        elif order_helps:
            verdict = "mixed"
            answer = ("Temporal order gives a significant gain over the order-free Set Transformer, but the overall "
                      "improvement of the Temporal Transformer over the mean-pooling family is not significant. "
                      "Explicit temporal modelling therefore offers at most a modest benefit; mean pooling remains a "
                      "strong, sufficient baseline at a fraction of the complexity.")
        else:
            verdict = "mean_pool_sufficient"
            answer = ("No temporal model significantly outperforms mean pooling after correction for multiple "
                      "comparisons. Mean pooling of CLIP frame embeddings is sufficient for this text-to-video "
                      "retrieval setting: MSR-VTT captions are dominated by static appearance cues (objects, "
                      "scenes, people) that are already captured by averaging frames.")
    else:
        answer = ("Learned temporal heads were not trained in this run, so only parameter-free aggregators can be "
                  "compared; see the frame-sweep and order-ablation sections.")

    xc = t2v.get("xclip")
    if xc:
        if xc["MRR"] < mp["MRR"] - 0.05:
            pts.append(f"The off-the-shelf spatio-temporal model X-CLIP (fine-tuned on Kinetics-400 action labels) reaches "
                       f"only R@1={xc['R@1']:.1f}% / MRR={xc['MRR']:.3f} (median rank {xc['MdR']:.0f}) versus "
                       f"{mp['R@1']:.1f}% / {mp['MRR']:.3f} for mean-pooled CLIP. Its text tower was adapted to ~400 "
                       f"short action-class names, so it does not transfer to free-form MSR-VTT captions: a model "
                       f"that natively encodes time is not automatically better - large-scale image-text alignment "
                       f"matters more than temporal architecture for this task.")
        else:
            pts.append(f"The off-the-shelf spatio-temporal model X-CLIP reaches R@1={xc['R@1']:.1f}% / MRR={xc['MRR']:.3f} "
                       f"versus {mp['R@1']:.1f}% / {mp['MRR']:.3f} for mean-pooled CLIP"
                       + (" - a native video architecture gives a measurable advantage." if xc["MRR"] > mp["MRR"] + 0.02
                          else " - no clear advantage from a native video architecture."))
    pc = R.get("positive_control", {}).get("mrr")
    if pc:
        pts.append(f"Positive control: on a synthetic task where only frame ORDER distinguishes clips, mean pooling "
                   f"scores MRR {pc['mean_pool']:.3f} and the order-free Set Transformer {pc['set_tfm']:.3f}, while the "
                   f"Temporal Transformer reaches {pc['temporal_tfm']:.3f} and the LSTM {pc['seq_lstm']:.3f}. The temporal "
                   f"heads and the evaluation are therefore capable of detecting a temporal benefit when one exists; "
                   f"any lack of gain on MSR-VTT reflects the data, not the models.")
    pts.append("Caveat: with 200 queries a 1-query change is 0.5 R@1 points, so small differences fall inside the "
               "95% bootstrap confidence intervals; conclusions are based on paired tests, not raw deltas.")
    return {"verdict": verdict, "answer": answer, "points": pts}

#### `mvr/figures.py`
**Figures** – matplotlib charts used in this notebook and the report.

In [ ]:
%%writefile mvr/figures.py
"""Static figures (PNG) for the technical report. The dashboard draws its own
interactive versions of the same charts with Altair."""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

from .config import LEARNED_METHODS, PARAM_FREE_METHODS, method_label  # noqa: E402

# reference categorical palette (fixed order), text inks, surface
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
FAMILY_COLOR = {"param_free": BLUE, "learned": ORANGE, "external": AQUA}


def family(m: str) -> str:
    return "param_free" if m in PARAM_FREE_METHODS else ("learned" if m in LEARNED_METHODS else "external")


def _style(ax, title, ylabel=None):
    ax.set_title(title, loc="left", fontsize=11, color=INK, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, color=INK2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=8)
    ax.yaxis.grid(True, color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


def _legend_family(ax, present):
    from matplotlib.patches import Patch
    names = {"param_free": "Parameter-free (zero-shot)", "learned": "Learned head (3 seeds, mean ± sd)",
             "external": "External video model"}
    ax.legend(handles=[Patch(color=FAMILY_COLOR[f], label=names[f]) for f in present],
              frameon=False, fontsize=8, loc="upper left")


def fig_main_bars(R, out: Path):
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    t2v, sd = R["t2v"], R["t2v_std"]
    ms = list(t2v)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax, key, unit in ((axes[0], "R@1", "%"), (axes[1], "MRR", "")):
        vals = [t2v[m][key] for m in ms]
        err = [sd.get(m, {}).get(key, 0) for m in ms]
        cols = [FAMILY_COLOR[family(m)] for m in ms]
        bars = ax.bar(range(len(ms)), vals, yerr=err, color=cols, width=0.62, capsize=3,
                      error_kw={"elinewidth": 1, "ecolor": INK2})
        ax.axhline(t2v["mean_pool"][key], color=INK2, linestyle="--", linewidth=1)
        top = max(v + e for v, e in zip(vals, err))
        ax.set_ylim(0, top * 1.15 if top > 0 else 1)
        for b, v, e in zip(bars, vals, err):
            ax.text(b.get_x() + b.get_width() / 2, v + e + top * 0.01, f"{v:.1f}" if unit else f"{v:.3f}",
                    ha="center", va="bottom", fontsize=7, color=INK)
        ax.set_xticks(range(len(ms)))
        ax.set_xticklabels([method_label(m).replace(" (", "\n(") for m in ms], rotation=35, ha="right", fontsize=7)
        _style(ax, f"Text-to-video {key}", key + (" (%)" if unit else ""))
    names = {"param_free": "Parameter-free (zero-shot)", "learned": "Learned head (mean ± sd over seeds)",
             "external": "External video model"}
    present = [f for f in ("param_free", "learned", "external") if any(family(m) == f for m in ms)]
    handles = [Patch(color=FAMILY_COLOR[f], label=names[f]) for f in present]
    handles.append(Line2D([0], [0], color=INK2, linestyle="--", label="Mean-pool baseline"))
    fig.legend(handles=handles, loc="upper center", ncol=len(handles), frameon=False, fontsize=8,
               bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(out, dpi=160)
    plt.close(fig)


def fig_frame_sweep(R, out: Path):
    sw = R["frame_sweep"]
    show = [m for m in ("mean_pool", "max_pool", "query_attn", "temporal_tfm") if m in sw]
    cols = [BLUE, ORANGE, AQUA, YELLOW]
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for m, c in zip(show, cols):
        ks = sorted(sw[m], key=int)
        ax.plot([int(k) for k in ks], [sw[m][k]["MRR"] for k in ks], marker="o", markersize=5, linewidth=2,
                color=c, label=method_label(m))
    ax.set_xscale("log", base=2)
    ax.set_xticks([int(k) for k in sorted(sw[show[0]], key=int)])
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel("Frames sampled per clip", color=INK2)
    _style(ax, "MRR vs number of sampled frames", "MRR")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def fig_order(R, out: Path):
    oa = R["order_ablation"]
    ms = list(oa)
    conds, cols = ["original", "shuffled", "reversed"], [BLUE, ORANGE, AQUA]
    x = np.arange(len(ms))
    w = 0.26
    fig, ax = plt.subplots(figsize=(10, 4))
    for i, (c, col) in enumerate(zip(conds, cols)):
        ax.bar(x + (i - 1) * w, [oa[m][c]["MRR"] for m in ms], width=w - 0.02, color=col, label=f"{c} frame order")
    ax.set_xticks(x)
    ax.set_xticklabels([method_label(m).replace(" (", "\n(") for m in ms], rotation=30, ha="right", fontsize=7)
    _style(ax, "Temporal-order ablation: MRR with original / shuffled / reversed frames", "MRR")
    ax.title.set_position((0, 1.14))
    lo = min(oa[m][c]["MRR"] for m in ms for c in conds)
    hi = max(oa[m][c]["MRR"] for m in ms for c in conds)
    ax.set_ylim(max(0, lo - 0.08), hi * 1.12)
    ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.06))
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def fig_per_category(R, out: Path):
    pc = R["per_category"]
    ms = [m for m in pc]
    cats = list(next(iter(pc.values())).keys())
    M = np.array([[pc[m][c]["R@5"] for m in ms] for c in cats])
    fig, ax = plt.subplots(figsize=(1.0 * len(ms) + 3, 7))
    im = ax.imshow(M, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(ms)))
    ax.set_xticklabels([method_label(m) for m in ms], rotation=40, ha="right", fontsize=7)
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels(cats, fontsize=8)
    for i in range(len(cats)):
        for j in range(len(ms)):
            ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=6.5,
                    color="white" if M[i, j] > 60 else INK)
    ax.set_title("Per-category R@5 (%) - 10 queries per category", loc="left", fontsize=11, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.03)
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def fig_training(R, out: Path):
    tr = R.get("training", {})
    show = [(a, c) for a, c in zip(LEARNED_METHODS, [BLUE, ORANGE, AQUA, YELLOW]) if f"{a}_seed0" in tr]
    if not show:
        return
    fig, ax = plt.subplots(figsize=(6.4, 4))
    for a, c in show:
        h = tr[f"{a}_seed0"]
        ax.plot(h["epoch"], h["val_mrr"], color=c, linewidth=2, label=method_label(a))
    ax.set_xlabel("Epoch (0 = initialisation = mean pooling)", color=INK2)
    _style(ax, "Validation MRR during training (held-out TRAIN clips, seed 0)", "Val MRR")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)


def fig_architecture(out: Path):
    """Block diagram of the retrieval system."""
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
    fig, ax = plt.subplots(figsize=(12, 5.6))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 6)
    ax.axis("off")

    def box(x, y, w, h, text, fc):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04,rounding_size=0.12",
                                    fc=fc, ec=INK2, lw=1))
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=8, color=INK, wrap=True)

    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=12, color=INK2, lw=1.2))

    off = "#dbe8f8"; tx = "#fde3d8"; idx = "#d3f1e6"; ui = "#fbefcf"
    ax.text(0.1, 5.75, "OFFLINE (index build)", fontsize=9, fontweight="bold", color=INK2)
    box(0.1, 4.3, 1.8, 1.1, "MSR-VTT\n100 clips\n(5 x 20 categories)", off)
    box(2.3, 4.3, 1.8, 1.1, "Frame sampling\nT=16 segment\nmid-points (OpenCV)", off)
    box(4.5, 4.3, 1.8, 1.1, "CLIP ViT image\nencoder (frozen)\n+ projection", off)
    box(6.7, 4.3, 2.2, 1.1, "Temporal aggregation\nmean / max / single /\nTransformer / LSTM", off)
    box(9.3, 4.3, 2.5, 1.1, "Video index\nL2-norm vectors\nFAISS IndexFlatIP", idx)
    arrow(1.9, 4.85, 2.3, 4.85); arrow(4.1, 4.85, 4.5, 4.85); arrow(6.3, 4.85, 6.7, 4.85); arrow(8.9, 4.85, 9.3, 4.85)

    ax.text(0.1, 3.55, "TRAINING (disjoint MSR-VTT train clips)", fontsize=9, fontweight="bold", color=INK2)
    box(4.5, 2.6, 1.8, 0.8, "Train clips x 20\ncaptions (frozen CLIP)", tx)
    box(6.7, 2.6, 2.2, 0.8, "InfoNCE on temporal\nheads (3 seeds)", tx)
    arrow(6.3, 3.0, 6.7, 3.0); arrow(7.8, 3.4, 7.8, 4.3)

    ax.text(0.1, 1.85, "ONLINE (query time)", fontsize=9, fontweight="bold", color=INK2)
    box(0.1, 0.4, 1.8, 1.1, "Streamlit UI\nnatural-language\nquery", ui)
    box(2.3, 0.4, 1.8, 1.1, "Query processing\nnormalise, spell-fix,\ntemplates, chunking", ui)
    box(4.5, 0.4, 1.8, 1.1, "CLIP text encoder\n(frozen) +\nprojection", ui)
    box(6.7, 0.4, 2.2, 1.1, "Shared 512-d space\ncosine = dot product\n(query-attn optional)", idx)
    box(9.3, 0.4, 2.5, 1.1, "Top-K clips + scores\nplayable videos,\nper-frame similarity", ui)
    arrow(1.9, 0.95, 2.3, 0.95); arrow(4.1, 0.95, 4.5, 0.95); arrow(6.3, 0.95, 6.7, 0.95); arrow(8.9, 0.95, 9.3, 0.95)
    arrow(10.55, 4.3, 10.55, 1.5)
    ax.text(10.65, 2.9, "search", fontsize=8, color=INK2)
    fig.tight_layout()
    fig.savefig(out, dpi=170)
    plt.close(fig)


def make_all_figures(R: dict, d: Path) -> dict:
    d.mkdir(parents=True, exist_ok=True)
    out = {}
    for name, fn in (("main_bars", fig_main_bars), ("frame_sweep", fig_frame_sweep), ("order", fig_order),
                     ("per_category", fig_per_category), ("training", fig_training)):
        p = d / f"{name}.png"
        try:
            fn(R, p)
            if p.exists():
                out[name] = p
        except Exception as e:
            print(f"figure {name} failed: {e}")
    p = d / "architecture.png"
    fig_architecture(p)
    out["architecture"] = p
    return out

#### `mvr/pipeline.py`
**Pipeline** – data → features → heads → index → X-CLIP → evaluation, each step cached on disk.

In [ ]:
%%writefile mvr/pipeline.py
"""End-to-end pipeline: data -> features -> temporal heads -> index -> evaluation.

Each step caches its output on disk, so the dashboard can resume and the
search page loads instantly once the pipeline has run once.
"""
from __future__ import annotations

import json
import re
import time
import traceback
from collections import Counter
from pathlib import Path
from typing import Callable, Optional

import numpy as np

from .config import LEARNED_METHODS, PARAM_FREE_METHODS, Config
from .data import load_subset, prepare_subset, read_video_frames, segment_indices
from .index import VideoIndex
from .metrics import holm, paired_bootstrap, summarize_ranks, t2v_ranks, v2t_metrics, wilcoxon_p
from .temporal import (head_embed, load_head, pool_video, query_attn_scores, save_head,
                       select_tau, train_head)

ProgressFn = Optional[Callable[[float, str], None]]

TEMPORAL_WORDS = {"then", "after", "before", "while", "begins", "begin", "starts", "start", "started",
                  "finally", "first", "again", "continues", "continue", "until", "during", "later",
                  "next", "followed", "afterwards", "meanwhile", "eventually", "once", "now"}


def _p(progress: ProgressFn, frac, msg):
    if progress:
        progress(min(max(frac, 0.0), 1.0), msg)
    else:
        print(f"[{frac*100:5.1f}%] {msg}", flush=True)


_ENCODER_CACHE: dict = {}


def get_encoder(cfg: Config):
    from .encoders import ClipEncoder
    key = (cfg.clip_model, cfg.resolved_device())
    if key not in _ENCODER_CACHE:
        _ENCODER_CACHE.clear()
        _ENCODER_CACHE[key] = ClipEncoder(cfg.clip_model, cfg.resolved_device())
    return _ENCODER_CACHE[key]


# ---------------------------------------------------------------------------
# Step 2: feature extraction
# ---------------------------------------------------------------------------
def step_features(cfg: Config, progress: ProgressFn = None, force: bool = False) -> dict:
    subset = load_subset(cfg)
    if cfg.features_path.exists() and not force:
        cached = dict(np.load(cfg.features_path, allow_pickle=True))
        if subset is not None and _same_ids(cached, subset):
            _p(progress, 1.0, "Features already extracted (cached)")
            return cached
        _p(progress, 0.0, "Index/training clips changed since the cached features were computed - re-extracting")
    if subset is None:
        raise RuntimeError("Run the data step first.")
    enc = get_encoder(cfg)
    T = cfg.num_frames

    # Per-clip cache: every encoded clip is saved immediately, so an interrupted run (e.g. a lab
    # session timeout) resumes from where it stopped instead of starting over.
    cache_dir = Path(cfg.artifacts_dir) / "clip_cache" / f"{cfg.model_tag}_T{T}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    timing_file = cache_dir / "timing.json"
    timing = json.loads(timing_file.read_text()) if timing_file.exists() else {"n": 0, "dec": 0.0, "enc": 0.0}

    def encode_videos(items, lo, hi, label):
        feats, ok_ids, n_cached = [], [], 0
        for i, v in enumerate(items):
            cf = cache_dir / f"{v['video_id']}.npy"
            if cf.exists():
                feats.append(np.load(cf))
                ok_ids.append(i)
                n_cached += 1
                continue
            path = cfg.videos_dir / f"{v['video_id']}.mp4"
            try:
                t0 = time.perf_counter()
                frames, _ = read_video_frames(path, T)
                t1 = time.perf_counter()
                f = enc.encode_frames(frames, cfg.batch_size).astype(np.float32)
                timing["enc"] += time.perf_counter() - t1
                timing["dec"] += t1 - t0
                timing["n"] += 1
                np.save(cf, f)
                feats.append(f)
                ok_ids.append(i)
            except Exception:
                continue
            if i % 5 == 0:
                timing_file.write_text(json.dumps(timing))
                _p(progress, lo + (hi - lo) * (i + 1) / len(items),
                   f"{label}: {i+1}/{len(items)} clips ({n_cached} from cache)")
        timing_file.write_text(json.dumps(timing))
        return np.stack(feats) if feats else np.zeros((0, T, enc.dim), np.float32), ok_ids

    index_frames, ok = encode_videos(subset["index"], 0.0, 0.35, "Encoding index clip frames")
    if len(ok) != len(subset["index"]):
        raise RuntimeError("Some index videos could not be decoded; re-run the data step with force.")
    _p(progress, 0.36, "Encoding the 200 text queries")
    t0 = time.perf_counter()
    query_emb = enc.encode_text([q["text"] for q in subset["queries"]])
    t_text = time.perf_counter() - t0

    train_frames, ok_tr = encode_videos(subset["train"], 0.38, 0.85, "Encoding training clip frames")
    train_items = [subset["train"][i] for i in ok_tr]
    C = 20
    train_text = np.zeros((len(train_items), C, enc.dim), np.float32)
    text_mask = np.zeros((len(train_items), C), bool)
    todo = []
    for i, t in enumerate(train_items):
        cf = cache_dir / f"{t['video_id']}_captions.npy"
        n = len(t["captions"][:C])
        if cf.exists():
            E = np.load(cf)
            train_text[i, :len(E)] = E
            text_mask[i, :len(E)] = True
        else:
            todo.append((i, t, n))
    for k in range(0, len(todo), 100):   # 100 clips (~2000 captions) per batch, cached per clip
        _p(progress, 0.86 + 0.13 * k / max(len(todo), 1), f"Encoding training captions: clip {k}/{len(todo)}")
        chunk = todo[k:k + 100]
        flat = [c for _, t, n in chunk for c in t["captions"][:n]]
        E = enc.encode_text(flat)
        pos = 0
        for i, t, n in chunk:
            e = E[pos:pos + n].astype(np.float32)
            pos += n
            np.save(cache_dir / f"{t['video_id']}_captions.npy", e)
            train_text[i, :n] = e
            text_mask[i, :n] = True
    n_enc = max(timing["n"], 1)
    t_dec, t_enc = timing["dec"] / n_enc, timing["enc"] / n_enc

    out = {"index_ids": np.array([v["video_id"] for v in subset["index"]]),
           "query_texts": np.array([q["text"] for q in subset["queries"]]),
           "index_frames": index_frames.astype(np.float32), "query_emb": query_emb.astype(np.float32),
           "train_frames": train_frames.astype(np.float32), "train_text": train_text, "text_mask": text_mask,
           "train_ids": np.array([t["video_id"] for t in train_items]),
           "timing": np.array(json.dumps({
               "decode_s_per_video": t_dec, "encode_s_per_video": t_enc,
               "text_encode_ms_per_query": 1000 * t_text / len(subset["queries"]),
               "device": cfg.resolved_device(), "model": cfg.clip_model, "frames_per_video": T}))}
    cfg.features_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cfg.features_path, **out)
    _p(progress, 1.0, f"Saved features to {cfg.features_path.name}")
    return out


def _same_ids(F: dict, subset: dict) -> bool:
    """Cached features are valid only for exactly the same index clips, queries and training clips."""
    try:
        return (list(F["index_ids"]) == [v["video_id"] for v in subset["index"]]
                and list(F["query_texts"]) == [q["text"] for q in subset["queries"]]
                and set(F["train_ids"]) <= {t["video_id"] for t in subset["train"]})
    except KeyError:
        return False


def load_features(cfg: Config) -> dict | None:
    if not cfg.features_path.exists():
        return None
    return dict(np.load(cfg.features_path, allow_pickle=True))


# ---------------------------------------------------------------------------
# Step 3: learned temporal heads
# ---------------------------------------------------------------------------
def head_path(cfg: Config, arch: str, seed: int) -> Path:
    return cfg.models_dir / f"{arch}_T{cfg.num_frames}_seed{seed}.pt"


def step_train(cfg: Config, progress: ProgressFn = None, force: bool = False) -> dict:
    F = load_features(cfg)
    if F is None:
        raise RuntimeError("Run the feature step first.")
    if len(F["train_frames"]) < 40:
        raise RuntimeError("Training pool too small (need >= 40 clips). Increase 'train videos per category'.")
    dev = cfg.resolved_device()
    tau, tau_curve = select_tau(F["train_frames"], F["train_text"], F["text_mask"])
    cfg.models_dir.mkdir(parents=True, exist_ok=True)
    # heads must be retrained if the training pool changed (e.g. clips removed by the content filter)
    train_ids = sorted(str(x) for x in F["train_ids"])
    pool_file = cfg.models_dir / f"train_pool_T{cfg.num_frames}.json"
    if not force and (not pool_file.exists() or json.loads(pool_file.read_text()) != train_ids):
        if any(cfg.models_dir.glob(f"*_T{cfg.num_frames}_seed*.pt")):
            _p(progress, 0.0, "Training pool changed since the cached heads were trained - retraining")
        force = True
    (cfg.models_dir / "tau.json").write_text(json.dumps({"tau": tau, "grid_mrr": tau_curve}))
    jobs = [(a, s) for a in LEARNED_METHODS for s in cfg.train_seeds]
    hists = {}
    for j, (arch, seed) in enumerate(jobs):
        p = head_path(cfg, arch, seed)
        if p.exists() and not force:
            hists[f"{arch}_seed{seed}"] = json.loads(p.with_suffix(".json").read_text())
            continue
        lo, hi = j / len(jobs), (j + 1) / len(jobs)
        head, hist = train_head(arch, F["train_frames"], F["train_text"], F["text_mask"], seed=seed,
                                epochs=cfg.eff_epochs(), lr=cfg.lr, weight_decay=cfg.weight_decay,
                                batch=cfg.train_batch, layers=cfg.tfm_layers, heads=cfg.tfm_heads, device=dev,
                                progress=lambda f, m: _p(progress, lo + (hi - lo) * f, m))
        save_head(head, hist, p)
        hists[f"{arch}_seed{seed}"] = hist
    pool_file.write_text(json.dumps(train_ids))   # written only after ALL heads match this pool
    _p(progress, 1.0, f"Trained {len(jobs)} temporal heads (query-attention tau={tau})")
    return hists


def load_heads(cfg: Config, dim: int) -> dict:
    heads = {}
    for arch in LEARNED_METHODS:
        for s in cfg.train_seeds:
            p = head_path(cfg, arch, s)
            if p.exists():
                heads[(arch, s)] = load_head(arch, dim, p, cfg.num_frames, cfg.tfm_layers, cfg.tfm_heads, "cpu")
    return heads


def get_tau(cfg: Config) -> float:
    p = cfg.models_dir / "tau.json"
    return json.loads(p.read_text())["tau"] if p.exists() else 0.01


# ---------------------------------------------------------------------------
# Step 4: index
# ---------------------------------------------------------------------------
def step_index(cfg: Config, progress: ProgressFn = None) -> VideoIndex:
    F = load_features(cfg)
    subset = load_subset(cfg)
    fr = F["index_frames"]
    t0 = time.perf_counter()
    pooled = {m: pool_video(fr, m) for m in ("single_frame", "mean_pool", "max_pool")}
    heads = load_heads(cfg, fr.shape[-1])
    for arch in LEARNED_METHODS:
        if (arch, cfg.train_seeds[0]) in heads:
            pooled[arch] = head_embed(heads[(arch, cfg.train_seeds[0])], fr)
    idx = VideoIndex([v["video_id"] for v in subset["index"]], fr, pooled, tau=get_tau(cfg))
    build_ms = (time.perf_counter() - t0) * 1000
    idx.save(cfg.index_dir)
    (cfg.index_dir / "build.json").write_text(json.dumps({"build_ms": build_ms, "backend": idx.backend}))
    _p(progress, 1.0, f"Index built with {len(pooled)} representations ({idx.backend}, {build_ms:.1f} ms)")
    return idx


def load_index(cfg: Config) -> VideoIndex | None:
    if (cfg.index_dir / "meta.json").exists():
        return VideoIndex.load(cfg.index_dir)
    return None


# ---------------------------------------------------------------------------
# Step 5 (optional): X-CLIP external spatio-temporal reference
# ---------------------------------------------------------------------------
def step_xclip(cfg: Config, progress: ProgressFn = None, force: bool = False) -> str:
    out = Path(cfg.artifacts_dir) / "xclip_scores.npy"
    ids_file = out.with_suffix(".ids.json")
    cur = load_subset(cfg)
    sig = {"index": [v["video_id"] for v in cur["index"]], "queries": [q["text"] for q in cur["queries"]]} if cur else None
    if out.exists() and not force and ids_file.exists() and json.loads(ids_file.read_text()) == sig:
        return "cached"
    if not cfg.use_xclip:
        return "disabled"
    try:
        from .encoders import XClipScorer
        subset = load_subset(cfg)
        _p(progress, 0.05, f"Loading {cfg.xclip_model}")
        xs = XClipScorer(cfg.xclip_model, cfg.resolved_device())
        pix = []
        for i, v in enumerate(subset["index"]):
            frames, _ = read_video_frames(cfg.videos_dir / f"{v['video_id']}.mp4", xs.num_frames)
            pix.append(xs.video_pixels(frames))
            if i % 10 == 0:
                _p(progress, 0.1 + 0.6 * i / len(subset["index"]), f"X-CLIP decoding {i}/{len(subset['index'])}")
        _p(progress, 0.75, "X-CLIP scoring 200 queries x 100 videos")
        S = xs.score_matrix([q["text"] for q in subset["queries"]], pix)
        np.save(out, S)
        ids_file.write_text(json.dumps(sig))
        (Path(cfg.artifacts_dir) / "xclip_error.txt").unlink(missing_ok=True)
        _p(progress, 1.0, "X-CLIP done")
        return "ok"
    except Exception as e:
        (Path(cfg.artifacts_dir) / "xclip_error.txt").write_text(traceback.format_exc())
        _p(progress, 1.0, f"X-CLIP skipped: {e}")
        return f"failed: {e}"


# ---------------------------------------------------------------------------
# Step 6: evaluation & research investigation
# ---------------------------------------------------------------------------
def _scores_for(method: str, q: np.ndarray, frames: np.ndarray, heads: dict, seeds, tau: float,
                pos_idx=None) -> list[np.ndarray]:
    """Return a list of (Q,N) score matrices (one per seed for learned methods)."""
    if method in ("single_frame", "mean_pool", "max_pool"):
        return [q @ pool_video(frames, method).T]
    if method == "query_attn":
        return [query_attn_scores(q, frames, tau)[0]]
    return [q @ head_embed(heads[(method, s)], frames, pos_idx).T for s in seeds if (method, s) in heads]


def step_eval(cfg: Config, progress: ProgressFn = None) -> dict:
    subset = load_subset(cfg)
    F = load_features(cfg)
    fr, Q = F["index_frames"], F["query_emb"]
    gt = np.array([q["video_idx"] for q in subset["queries"]])
    cats = np.array([q["category"] for q in subset["queries"]])
    heads = load_heads(cfg, fr.shape[-1])
    seeds = list(cfg.train_seeds)
    tau = get_tau(cfg)
    methods = PARAM_FREE_METHODS + [m for m in LEARNED_METHODS if any((m, s) in heads for s in seeds)]
    ks = tuple(cfg.ks)
    R: dict = {"config": {**cfg.to_dict(), "train_videos_per_category": cfg.eff_train_per_cat(),
                          "epochs": cfg.eff_epochs()}, "methods": methods, "tau": tau,
               "n_queries": int(len(Q)), "n_videos": int(len(fr))}
    rel = [set() for _ in range(len(fr))]
    for qi, g in enumerate(gt):
        rel[g].add(qi)

    # ---------------- main T2V / V2T tables ----------------
    _p(progress, 0.05, "Computing T2V and V2T metrics for all representations")
    t2v, t2v_std, v2t, v2t_std, per_query = {}, {}, {}, {}, {}
    for m in methods:
        Ss = _scores_for(m, Q, fr, heads, seeds, tau)
        tm = [summarize_ranks(t2v_ranks(S, gt), ks) for S in Ss]
        vm = [v2t_metrics(S.T, rel, ks)[0] for S in Ss]
        t2v[m] = {k: float(np.mean([d[k] for d in tm])) for k in tm[0]}
        t2v_std[m] = {k: float(np.std([d[k] for d in tm])) for k in tm[0]}
        v2t[m] = {k: float(np.mean([d[k] for d in vm])) for k in vm[0]}
        v2t_std[m] = {k: float(np.std([d[k] for d in vm])) for k in vm[0]}
        ranks = np.stack([t2v_ranks(S, gt) for S in Ss])        # (seeds,Q)
        per_query[m] = {"rank_seed0": ranks[0].tolist(), "rr_mean": (1 / ranks).mean(0).tolist(),
                        **{f"hit@{k}": (ranks <= k).mean(0).tolist() for k in ks}}
    xs_path = Path(cfg.artifacts_dir) / "xclip_scores.npy"
    xs_ids = xs_path.with_suffix(".ids.json")
    xs_ok = xs_ids.exists() and json.loads(xs_ids.read_text()).get("index") == [v["video_id"] for v in subset["index"]]
    if xs_path.exists() and xs_ok:
        S = np.load(xs_path)
        if S.shape == (len(Q), len(fr)):
            t2v["xclip"] = summarize_ranks(t2v_ranks(S, gt), ks)
            v2t["xclip"] = v2t_metrics(S.T, rel, ks)[0]
            t2v_std["xclip"] = {k: 0.0 for k in t2v["xclip"]}
            v2t_std["xclip"] = {k: 0.0 for k in v2t["xclip"]}
            r = t2v_ranks(S, gt)
            per_query["xclip"] = {"rank_seed0": r.tolist(), "rr_mean": (1 / r).tolist(),
                                  **{f"hit@{k}": (r <= k).astype(float).tolist() for k in ks}}
    R.update({"t2v": t2v, "t2v_std": t2v_std, "v2t": v2t, "v2t_std": v2t_std, "per_query": per_query})

    # ---------------- significance tests ----------------
    _p(progress, 0.25, "Paired bootstrap + Wilcoxon significance tests")
    comparisons = [(m, "mean_pool") for m in per_query if m != "mean_pool"]
    comparisons += [("temporal_tfm", "set_tfm"), ("set_tfm", "mlp_adapter"),
                    ("temporal_tfm", "mlp_adapter"), ("seq_lstm", "mlp_adapter")]
    sig, pw = [], {}
    for a, b in comparisons:   # effect = a - b
        if a not in per_query or b not in per_query:
            continue
        pa, pb = per_query[a], per_query[b]
        row = {"method": a, "reference": b}
        row["MRR"] = paired_bootstrap(np.array(pb["rr_mean"]), np.array(pa["rr_mean"]), cfg.bootstrap_samples)
        for k in (1, 5):
            row[f"R@{k}"] = paired_bootstrap(np.array(pb[f"hit@{k}"]) * 100, np.array(pa[f"hit@{k}"]) * 100,
                                             cfg.bootstrap_samples)
        row["wilcoxon_p"] = wilcoxon_p(np.array(pb["rr_mean"]), np.array(pa["rr_mean"]))
        d = np.array(pa["rr_mean"]) - np.array(pb["rr_mean"])
        row["n_better"], row["n_worse"], row["n_tie"] = int((d > 1e-9).sum()), int((d < -1e-9).sum()), int((np.abs(d) <= 1e-9).sum())
        pw[f"{a}|{b}"] = row["wilcoxon_p"]
        sig.append(row)
    adj = holm(pw)
    for row in sig:
        row["wilcoxon_p_holm"] = adj.get(f"{row['method']}|{row['reference']}", float("nan"))
    R["significance"] = sig

    # ---------------- number-of-frames sweep ----------------
    _p(progress, 0.40, "Frame-count sweep")
    T = fr.shape[1]
    sweep = {}
    ks_frames = [k for k in (1, 2, 4, 8, 16, 32) if k <= T]
    for m in methods:
        sweep[m] = {}
        for k in ks_frames:
            sel = segment_indices(T, k)
            Ss = _scores_for(m, Q, fr[:, sel], heads, seeds, tau, pos_idx=sel)
            ms = [summarize_ranks(t2v_ranks(S, gt), ks) for S in Ss]
            sweep[m][str(k)] = {x: float(np.mean([d[x] for d in ms])) for x in ("R@1", "R@5", "MRR")}
    R["frame_sweep"] = sweep

    # ---------------- temporal order ablation (shuffle / reverse) ----------------
    _p(progress, 0.55, "Temporal-order ablation (shuffled / reversed frames)")
    rng = np.random.default_rng(123)
    perm = np.stack([rng.permutation(T) for _ in range(len(fr))])
    fr_shuf = np.take_along_axis(fr, perm[:, :, None], axis=1)
    fr_rev = fr[:, ::-1]
    order = {}
    for m in methods:
        if m == "single_frame":   # its "middle frame" changes under permutation - not meaningful here
            continue
        res = {}
        for name, X in (("original", fr), ("shuffled", fr_shuf), ("reversed", fr_rev)):
            Ss = _scores_for(m, Q, X, heads, seeds, tau)
            ms = [summarize_ranks(t2v_ranks(S, gt), ks) for S in Ss]
            res[name] = {x: float(np.mean([d[x] for d in ms])) for x in ("R@1", "R@5", "MRR")}
        order[m] = res
    R["order_ablation"] = order

    # ---------------- per-category & temporal-language subsets ----------------
    _p(progress, 0.65, "Per-category and temporal-language breakdown")
    names = subset["category_names"]
    per_cat = {}
    for m in per_query:
        rr = np.array(per_query[m]["rr_mean"])
        h5 = np.array(per_query[m]["hit@5"])
        per_cat[m] = {names[c]: {"MRR": float(rr[cats == c].mean()), "R@5": float(h5[cats == c].mean() * 100)}
                      for c in range(len(names)) if (cats == c).any()}
    R["per_category"] = per_cat
    is_temp = np.array([bool(TEMPORAL_WORDS & set(re.findall(r"[a-z]+", q["text"].lower())))
                        for q in subset["queries"]])
    R["temporal_language"] = {
        "n_temporal": int(is_temp.sum()), "n_other": int((~is_temp).sum()),
        "examples": [q["text"] for q, t in zip(subset["queries"], is_temp) if t][:8],
        "by_method": {m: {"temporal_MRR": float(np.array(per_query[m]["rr_mean"])[is_temp].mean()) if is_temp.any() else None,
                          "other_MRR": float(np.array(per_query[m]["rr_mean"])[~is_temp].mean())}
                      for m in per_query}}

    # ---------------- edge-case robustness (query side) ----------------
    _p(progress, 0.72, "Query edge-case robustness experiments")
    try:
        R["robustness"] = robustness_study(cfg, subset, fr, gt)
    except Exception as e:
        R["robustness"] = {"error": str(e)}

    # ---------------- positive control (can the setup detect temporal benefit?) ----------------
    _p(progress, 0.82, "Positive-control experiment (synthetic order-only task)")
    try:
        from .control import run_positive_control
        R["positive_control"] = run_positive_control()
    except Exception as e:
        R["positive_control"] = {"error": str(e)}

    # ---------------- efficiency & training curves ----------------
    _p(progress, 0.90, "Efficiency statistics")
    R["timing"] = json.loads(str(F["timing"]))
    bpath = cfg.index_dir / "build.json"
    if bpath.exists():
        R["timing"].update(json.loads(bpath.read_text()))
    idx = load_index(cfg)
    if idx is not None:
        lat = {}
        for m in idx.methods:
            t0 = time.perf_counter()
            for q in Q:
                idx.search(q, m, 10)
            lat[m] = 1000 * (time.perf_counter() - t0) / len(Q)
        R["timing"]["search_ms_per_query"] = lat
    curves = {}
    for arch in LEARNED_METHODS:
        for s in seeds:
            p = head_path(cfg, arch, s).with_suffix(".json")
            if p.exists():
                curves[f"{arch}_seed{s}"] = json.loads(p.read_text())
    R["training"] = curves
    tp = cfg.models_dir / "tau.json"
    if tp.exists():
        R["tau_selection"] = json.loads(tp.read_text())
    R["dataset"] = subset["protocol"]
    R["generated_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    from .analysis import research_conclusions
    R["conclusions"] = research_conclusions(R)
    cfg.results_path.write_text(json.dumps(R, indent=1, default=float))
    try:
        from .figures import make_all_figures
        make_all_figures(R, cfg.figures_dir)
    except Exception as e:
        print("figure generation failed:", e)
    _p(progress, 1.0, "Evaluation complete")
    return R


def robustness_study(cfg: Config, subset: dict, fr: np.ndarray, gt: np.ndarray) -> dict:
    """Evaluate query-side edge-case handling with the mean-pooling index."""
    from .query import QueryProcessor, inject_typos, shorten
    enc = get_encoder(cfg)
    lex = load_lexicon(cfg)
    qp = QueryProcessor(enc, lex)
    V = pool_video(fr, "mean_pool")
    texts = [q["text"] for q in subset["queries"]]
    rng = np.random.default_rng(7)
    typo = [inject_typos(t, rng) for t in texts]
    short = [shorten(t, 2) for t in texts]
    # "detailed" queries: the query caption + up to 9 other captions of the same clip (often > 77 tokens)
    detailed = []
    for q in subset["queries"]:
        caps = [c for c in subset["index"][q["video_idx"]]["all_captions"] if c != q["text"]][:9]
        detailed.append(". ".join([q["text"]] + caps) + ".")

    def run(ts, **kw):
        E = qp.encode_many(ts, **kw)
        return summarize_ranks(t2v_ranks(E @ V.T, gt), tuple(cfg.ks))

    off = dict(auto_correct=False, use_templates=False, use_chunking=False)
    rows = [
        ("Original captions", "none", run(texts, **off)),
        ("Short (2 content words)", "raw", run(short, **off)),
        ("Short (2 content words)", "prompt-template ensemble", run(short, auto_correct=False, use_templates=True)),
        ("Typo-corrupted", "raw", run(typo, **off)),
        ("Typo-corrupted", "spelling correction", run(typo, auto_correct=True, use_templates=False)),
        ("Detailed (multi-sentence)", "truncate at 77 tokens", run(detailed, **off)),
        ("Detailed (multi-sentence)", "chunk + average", run(detailed, auto_correct=False, use_templates=False, use_chunking=True)),
    ]
    n_long = sum(enc.token_count(d) > enc.max_len for d in detailed)
    n_unknown = sum(1 for t in typo if qp.process(t, auto_correct=False).unknown_words)
    return {"rows": [{"condition": a, "handling": b, **m} for a, b, m in rows],
            "n_long_queries": int(n_long), "n_typo_queries_with_unknown_words": int(n_unknown),
            "examples": {"short": short[:3], "typo": typo[:3], "detailed": detailed[:1]}}


def load_lexicon(cfg: Config) -> Counter:
    p = Path(cfg.data_dir) / "lexicon.json"
    return Counter(json.loads(p.read_text())) if p.exists() else Counter()


def load_results(cfg: Config) -> dict | None:
    return json.loads(cfg.results_path.read_text()) if cfg.results_path.exists() else None


def run_all(cfg: Config, progress: ProgressFn = None, force: bool = False) -> dict:
    steps = [("Data ingestion", lambda p: prepare_subset(cfg, p, force)),
             ("Feature extraction", lambda p: step_features(cfg, p, force)),
             ("Temporal head training", lambda p: step_train(cfg, p, force)),
             ("Index build", lambda p: step_index(cfg, p)),
             ("X-CLIP reference", lambda p: step_xclip(cfg, p, force)),
             ("Evaluation", lambda p: step_eval(cfg, p))]
    out = None
    for i, (name, fn) in enumerate(steps):
        lo, hi = i / len(steps), (i + 1) / len(steps)
        out = fn(lambda f, m, lo=lo, hi=hi, name=name: _p(progress, lo + (hi - lo) * f, f"[{name}] {m}"))
    return out

#### `mvr/report.py`
**Report** – builds the technical report (.docx) from the results.

In [ ]:
%%writefile mvr/report.py
"""Generates the Technical Report (.docx) directly from the evaluation results,
so every number, table and chart in the report comes from the actual run."""
from __future__ import annotations

import json
from pathlib import Path

from .config import LEARNED_METHODS, METHOD_INFO, Config, method_label
from .figures import make_all_figures


TEAM = [("TILAK", "2025AE05975", "System design, data pipeline, CLIP features, FAISS retrieval, lab execution"),
        ("OM PRAKASH UNIYAL", "2025AE05324", "Temporal models (pooling, Transformer, LSTM) and research ablation study"),
        ("KIRAN P", "2025AE05409", "Evaluation metrics, significance tests and results analysis"),
        ("GAURISHA MATHUR", "2025AE05642", "Streamlit dashboard and query edge-case handling"),
        ("KARAN RUPANI", "2025AE05813", "Experiments write-up and technical report")]


def build_report(cfg: Config, out: Path) -> Path:
    from docx import Document
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn
    from docx.shared import Cm, Pt

    R = json.loads(cfg.results_path.read_text())
    subset = json.loads(cfg.subset_path.read_text())
    figs = make_all_figures(R, cfg.figures_dir)
    C = R["conclusions"]

    doc = Document()
    sec = doc.sections[0]
    sec.left_margin = sec.right_margin = Cm(2.0)
    sec.top_margin = sec.bottom_margin = Cm(1.8)
    st = doc.styles["Normal"]
    st.font.name = "Calibri"
    st.font.size = Pt(10.5)

    def shade(cell, hexcol):
        tcPr = cell._tc.get_or_add_tcPr()
        shd = OxmlElement("w:shd")
        shd.set(qn("w:val"), "clear")
        shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"), hexcol)
        tcPr.append(shd)

    def table(header, rows, widths=None, font=8.5):
        t = doc.add_table(rows=1, cols=len(header))
        t.style = "Table Grid"
        t.alignment = WD_TABLE_ALIGNMENT.CENTER
        for i, h in enumerate(header):
            c = t.rows[0].cells[i]
            c.text = ""
            r = c.paragraphs[0].add_run(str(h))
            r.bold = True
            r.font.size = Pt(font)
            shade(c, "DBE8F8")
        for row in rows:
            cells = t.add_row().cells
            for i, v in enumerate(row):
                cells[i].text = ""
                r = cells[i].paragraphs[0].add_run(str(v))
                r.font.size = Pt(font)
        if widths:
            t.autofit = False
            for i, w in enumerate(widths):
                t.columns[i].width = Cm(w)
            for row in t.rows:
                for i, w in enumerate(widths):
                    row.cells[i].width = Cm(w)
        doc.add_paragraph()
        return t

    def fig(key, caption, width=16.5):
        p = figs.get(key)
        if p and Path(p).exists():
            doc.add_picture(str(p), width=Cm(width))
            doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
            cp = doc.add_paragraph(caption)
            cp.alignment = WD_ALIGN_PARAGRAPH.CENTER
            cp.runs[0].italic = True
            cp.runs[0].font.size = Pt(9)

    def bullets(items):
        for it in items:
            doc.add_paragraph(it, style="List Bullet")

    def para(text, bold_prefix=None):
        p = doc.add_paragraph()
        if bold_prefix:
            p.add_run(bold_prefix).bold = True
        p.add_run(text)
        return p

    t2v, sd, v2t = R["t2v"], R["t2v_std"], R["v2t"]
    mp = t2v["mean_pool"]

    # ------------------------------------------------------------------ cover
    h = doc.add_heading("Assignment 2: Natural-Language Video Retrieval Using MSR-VTT", 0)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p = doc.add_paragraph("Text-to-Video Retrieval Using Semantic and Temporal Video Representations")
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.runs[0].font.size = Pt(13)
    p = doc.add_paragraph("Course: Multimodal Information Retrieval  ·  Environment: BITS Virtual Lab")
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    table(["Group No.", "Member Name", "BITS ID", "Contribution"],
          [["10" if k == 0 else "", n, i, c] for k, (n, i, c) in enumerate(TEAM)], [2.2, 4.3, 3, 7.5])
    para(f"Results generated: {R['generated_at']}  ·  backbone: {R['config']['clip_model']}  ·  "
         f"{R['config']['num_frames']} frames/clip  ·  device: {R.get('timing', {}).get('device', 'n/a')}")

    doc.add_heading("Executive summary", 1)
    para(f"We built an end-to-end text-to-video retrieval system over a 100-clip MSR-VTT index (5 clips × 20 "
         f"categories) queried by 200 natural-language captions (a 200 × 100 search matrix), with a Streamlit dashboard "
         f"for arbitrary queries, playable Top-K results and live evaluation. Frames are encoded with a frozen CLIP "
         f"ViT and mapped, together with the text query, into CLIP's shared 512-d embedding space. With mean-pooled "
         f"frame embeddings the system reaches R@1 = {mp['R@1']:.1f}%, R@5 = {mp['R@5']:.1f}%, R@10 = {mp['R@10']:.1f}%, "
         f"MRR = {mp['MRR']:.3f} and a median rank of {mp['MdR']:.0f}.")
    para(C["answer"], "Research Investigation 1 – answer: ")

    # ------------------------------------------------------------------ 1 data
    doc.add_heading("1. Dataset and evaluation protocol", 1)
    pr = R["dataset"]
    table(["Item", "Value"], [
        ["Source", "MSR-VTT (Xu et al., CVPR 2016) – Hugging Face mirror friedrichor/MSR-VTT"],
        ["Index size N", f"{pr['n_index']} clips = {pr['videos_per_category']} clips × 20 categories, drawn from the 1k-A test split"],
        ["Query set Q", f"{pr['n_queries']} queries = {pr['captions_per_video']} human captions per clip "
                        "(the official 1k-A caption + one randomly chosen other caption)"],
        ["Search corpus matrix", f"{pr['n_queries']} × {pr['n_index']}"],
        ["Training pool (learned heads only)", f"{pr['n_train']} clips from the 9k train split, all 20 captions; "
                                                 "disjoint from the index (asserted in code)"],
        ["Sampling", f"stratified by category, fixed seed {pr['seed']}; undecodable clips are skipped and replaced"],
        ["Content filter", f"clips whose captions use sexual/explicit wording are excluded from index and training pool "
                           f"({pr.get('n_excluded_by_content_filter', 0)} candidate clips skipped)"],
        ["Relevance", "T2V: the clip a caption describes (1 relevant / query). V2T: the 2 captions of a clip"],
    ], [5, 12])
    cats = {}
    for v in subset["index"]:
        cats.setdefault(v["category_name"], 0)
        cats[v["category_name"]] += 1
    para("Categories (5 clips each): " + ", ".join(cats))

    # ------------------------------------------------------------------ 2 architecture
    doc.add_heading("2. System architecture", 1)
    fig("architecture", "Figure 1 – Offline indexing, training and online query pipelines.", 17)
    table(["Module (file)", "Responsibility"], [
        ["mvr/data.py", "Download annotations/videos, stratified 5×20 sampling, OpenCV frame decoding (TSN segment mid-points), thumbnails"],
        ["mvr/encoders.py", "CLIP image/text encoders (frozen) → shared 512-d space; CLIP pre-processing; optional X-CLIP"],
        ["mvr/temporal.py", "Mean/max/single/query-attentive pooling; learned temporal heads (MLP, Set-TFM, Temporal-TFM, LSTM) + InfoNCE training"],
        ["mvr/index.py", "FAISS IndexFlatIP per representation (cosine via inner product of unit vectors) + frame-level matrix"],
        ["mvr/query.py", "Query normalisation, unknown-word detection, spelling correction, prompt templates, long-query chunking"],
        ["mvr/metrics.py", "R@K, MRR, mAP, nDCG@10, MdR, MnR; paired bootstrap, Wilcoxon, Holm correction"],
        ["mvr/pipeline.py", "Orchestration with on-disk caching; evaluation & research experiments"],
        ["app.py", "Streamlit dashboard (pipeline runner, search, V2T, evaluation, research, dataset, report)"],
    ], [4, 13])

    # ------------------------------------------------------------------ 3 features
    doc.add_heading("3. Data ingestion and feature engineering", 1)
    tm = R.get("timing", {})
    bullets([
        f"Frame sampling: each clip is split into T = {R['config']['num_frames']} equal temporal segments and the "
        "middle frame of each segment is decoded (TSN-style uniform coverage of the whole clip). Frames are read "
        "sequentially with grab/retrieve for frame-accurate sampling; a full-decode fallback handles wrong frame-count metadata.",
        "Pre-processing: RGB conversion, bicubic resize of the short side to the model resolution, centre crop, "
        "normalisation with CLIP's channel mean/std.",
        f"Visual features: frozen CLIP ViT image tower + visual projection ({R['config']['clip_model']}), "
        "producing one L2-normalised 512-d embedding per frame (fp16 on GPU).",
        "Indexing: video-level vectors for every representation are stored in FAISS IndexFlatIP (exact inner-product "
        "search = cosine similarity on unit vectors); the (N × T × D) frame tensor is kept for query-conditioned pooling "
        "and per-frame similarity visualisation. Everything is cached to .npy/.faiss files.",
        f"Cost: decode {tm.get('decode_s_per_video', 0)*1000:.0f} ms/clip, CLIP encode "
        f"{tm.get('encode_s_per_video', 0)*1000:.0f} ms/clip, index build {tm.get('build_ms', 0):.1f} ms "
        f"({tm.get('backend', '')}).",
    ])

    # ------------------------------------------------------------------ 4 retrieval engine
    doc.add_heading("4. Cross-modal embedding and retrieval engine", 1)
    para("Queries are encoded by CLIP's text Transformer and text projection into the same 512-d space as the frames "
         "(CLIP was contrastively trained so that matching image/text pairs have high cosine similarity). "
         "Score(q, v) = cos(e_q, e_v) = e_q · e_v for unit vectors; the Top-K clips are returned with their scores. "
         "For query-attentive pooling, Score = Σ_t softmax_t(s_t/τ)·s_t with s_t = e_q · f_t "
         f"(τ = {R['tau']} chosen on the training pool, never on the test queries).")
    doc.add_heading("Representations compared", 2)
    table(["Representation", "Description", "Order-aware", "Trained"],
          [[method_label(m), METHOD_INFO[m][1], "yes" if m in ("temporal_tfm", "seq_lstm", "xclip") else "no",
            "yes" if m in LEARNED_METHODS else ("pre-trained" if m == "xclip" else "no")]
           for m in t2v], [4, 9.5, 1.8, 1.7])
    doc.add_heading("Edge-case handling", 2)
    table(["Edge case", "Handling in the engine"], [
        ["Empty / punctuation-only query", "Rejected with a helpful message (no meaningless search)"],
        ["Unicode noise, URLs, extra spaces", "NFKC normalisation, control-char removal, whitespace collapse"],
        ["Unknown / misspelled vocabulary", "Words not in CLIP's BPE vocabulary or the MSR-VTT training lexicon are "
                                            "flagged; frequency-ranked edit-distance (≤2) correction; BPE sub-words as fallback"],
        ["Very short queries (≤2 content words)", "Prompt-template ensemble: 'a video of {q}.', 'a video showing {q}.', '{q}'"],
        ["Long / detailed queries (>77 tokens)", "Sentence/clause-aware chunking, chunk embeddings averaged (no silent truncation)"],
        ["Non-English text", "Detected and flagged (CLIP text encoder is English-only)"],
        ["No good match in the collection", "Low-confidence warning when the top score is below the 5th percentile of correct-match scores"],
    ], [5, 12])
    rb = R.get("robustness", {})
    if "rows" in rb:
        para("Effect of the edge-case handling (mean-pool index, all 200 queries perturbed):")
        table(["Query condition", "Handling", "R@1", "R@5", "MRR", "MdR"],
              [[r["condition"], r["handling"], f"{r['R@1']:.1f}", f"{r['R@5']:.1f}", f"{r['MRR']:.3f}", f"{r['MdR']:.0f}"]
               for r in rb["rows"]], [4.5, 4.5, 1.6, 1.6, 1.8, 1.4])

    # ------------------------------------------------------------------ 5 dashboard
    doc.add_heading("5. Streamlit interactive dashboard", 1)
    bullets([
        "Pipeline Runner – every step (download, frame extraction, CLIP encoding, training, indexing, X-CLIP, "
        "evaluation) is launched from buttons with live progress; no manual script execution is required.",
        "Text → Video Search – free-text query, choice of representation, Top-K slider; results show playable "
        "clips, cosine scores, category, per-frame similarity profile, captions, query-processing diagnostics and latency.",
        "Video → Text – the reverse modality: pick a clip, retrieve its Top-K captions with relevance marks.",
        "Evaluation – metric tables (T2V and V2T), interactive charts, frame sweep, per-category heat-map, "
        "robustness and efficiency.",
        "Research Investigation – ablation ladder, significance tests, order ablation and conclusions.",
        "Dataset Browser and Report pages (this document is generated from the Report page).",
    ])

    # ------------------------------------------------------------------ 6 results
    doc.add_heading("6. Quantitative performance evaluation", 1)
    cols = ["R@1", "R@5", "R@10", "MRR", "mAP", "nDCG@10", "MdR", "MnR"]

    def fmt(m, c, tab, s):
        v, e = tab[m][c], s.get(m, {}).get(c, 0)
        if c.startswith("R@") or c in ("MdR", "MnR"):
            return f"{v:.1f}" + (f"±{e:.1f}" if e else "")
        return f"{v:.3f}" + (f"±{e:.3f}" if e else "")

    para("Table 1 – Text→Video retrieval (200 queries × 100 clips). Learned heads: mean ± sd over "
         f"{len(R['config']['train_seeds'])} seeds. With one relevant clip per query mAP equals MRR.", None)
    table(["Method"] + cols, [[method_label(m)] + [fmt(m, c, t2v, sd) for c in cols] for m in t2v],
          [4.2] + [1.6] * 8, 8)
    para("Table 2 – Video→Text retrieval (100 clips × 200 captions; 2 relevant captions per clip).")
    table(["Method"] + cols, [[method_label(m)] + [fmt(m, c, v2t, R["v2t_std"]) for c in cols] for m in v2t],
          [4.2] + [1.6] * 8, 8)
    fig("main_bars", "Figure 2 – T2V R@1 and MRR per representation (error bars: sd over seeds; dashed: mean-pool baseline).", 17)

    # ------------------------------------------------------------------ 7 research
    doc.add_heading("7. Research Investigation 1: is mean pooling sufficient?", 1)
    doc.add_heading("7.1 Experimental design", 2)
    para("To separate the effect of temporal modelling from confounds we use an ablation ladder in which each step "
         "adds exactly one capability on top of the same frozen CLIP frame features: (0) a single frame; (1) mean "
         "pooling; (1b) zero-shot max and query-attentive pooling; (2) an MLP adapter trained on in-domain data but "
         "with no frame interaction (control for domain adaptation); (3) a Set Transformer with frame–frame "
         "attention but no positional information (control for interaction without order); (4) a Temporal "
         "Transformer with learned temporal position embeddings and (4b) an LSTM – both explicitly model order. "
         "All learned heads are zero-initialised residuals of mean pooling, use the same data, loss (symmetric "
         "InfoNCE), optimiser (AdamW, one-cycle), temporal sub-sampling augmentation and early stopping on held-out "
         "training clips, and are run with multiple seeds. We additionally (i) shuffle/reverse frame order at test "
         "time, (ii) vary the number of frames, (iii) split queries by temporal language, and (iv) compare against "
         "X-CLIP, a natively spatio-temporal video-language model, when available.")
    doc.add_heading("7.2 Statistical tests", 2)
    para("Query-level paired bootstrap (10,000 resamples) 95% confidence intervals on ΔMRR and ΔR@K, Wilcoxon "
         "signed-rank tests on per-query reciprocal ranks, Holm–Bonferroni correction across comparisons.")
    table(["Comparison (A vs B)", "ΔMRR", "95% CI", "ΔR@1", "ΔR@5", "p (Wilcoxon)", "p (Holm)", "A better/worse/tie"],
          [[f"{method_label(r['method'])} vs {method_label(r['reference'])}", f"{r['MRR']['delta']:+.3f}",
            f"[{r['MRR']['ci_low']:+.3f}, {r['MRR']['ci_high']:+.3f}]", f"{r['R@1']['delta']:+.1f}",
            f"{r['R@5']['delta']:+.1f}", f"{r['wilcoxon_p']:.3g}", f"{r['wilcoxon_p_holm']:.3g}",
            f"{r['n_better']}/{r['n_worse']}/{r['n_tie']}"] for r in R["significance"]],
          [5, 1.4, 2.6, 1.3, 1.3, 1.6, 1.4, 2.3], 7.5)
    doc.add_heading("7.3 Temporal-order ablation and frame count", 2)
    oa = R["order_ablation"]
    table(["Method", "MRR original", "MRR shuffled", "MRR reversed", "R@1 original", "R@1 shuffled"],
          [[method_label(m), f"{oa[m]['original']['MRR']:.3f}", f"{oa[m]['shuffled']['MRR']:.3f}",
            f"{oa[m]['reversed']['MRR']:.3f}", f"{oa[m]['original']['R@1']:.1f}", f"{oa[m]['shuffled']['R@1']:.1f}"]
           for m in oa], [5, 2.4, 2.4, 2.4, 2.4, 2.4])
    fig("order", "Figure 3 – MRR when the frame order is shuffled or reversed at test time.", 16)
    fig("frame_sweep", "Figure 4 – MRR as a function of the number of sampled frames.", 12)
    tl = R["temporal_language"]
    para(f"Queries containing explicit temporal language ({tl['n_temporal']} of {tl['n_temporal'] + tl['n_other']}, "
         "e.g. 'then', 'after', 'starts'):")
    table(["Method", "MRR – temporal-language queries", "MRR – other queries"],
          [[method_label(m), f"{v['temporal_MRR']:.3f}" if v["temporal_MRR"] is not None else "–", f"{v['other_MRR']:.3f}"]
           for m, v in tl["by_method"].items()], [6, 5.5, 5.5])
    pcr = R.get("positive_control", {}).get("mrr")
    if pcr:
        para("Positive control – synthetic clips 'state A then state B' vs 'B then A' whose captions encode the "
             "ordered pair (only temporal order separates them). This verifies that the set-up can detect a "
             "temporal benefit when it exists:")
        table(["Method", "MRR (order-only task)"], [[method_label(m), f"{v:.3f}"] for m, v in pcr.items()], [8, 5])
    fig("per_category", "Figure 5 – Per-category R@5 (each cell = 10 queries).", 15)
    fig("training", "Figure 6 – Validation MRR during training; epoch 0 equals mean pooling.", 12)
    doc.add_heading("7.4 Findings and conclusion", 2)
    bullets(C["points"])
    para(C["answer"], "Conclusion: ")
    doc.add_heading("7.5 Insights", 2)
    bullets([
        "MSR-VTT captions mostly describe *what* is visible (people, objects, scenes, activities recognisable from "
        "single frames), so a frame-level image–text model already captures most of the retrievable signal; "
        "averaging frames adds robustness to unrepresentative frames.",
        "Gains from trained heads must be compared with an equally-trained, order-free control. Without the MLP "
        "adapter and Set-Transformer controls, domain-adaptation gains are easily misattributed to temporal modelling.",
        "Test-time frame shuffling is a cheap diagnostic of whether a 'temporal' model really uses order.",
        "Temporal modelling is most likely to matter for queries whose meaning depends on event order or motion "
        "direction (e.g. 'opens then closes', 'falls down'); such queries are rare in MSR-VTT, which limits the "
        "measurable benefit on this benchmark.",
    ])

    # ------------------------------------------------------------------ 8 efficiency / limits
    doc.add_heading("8. Efficiency", 1)
    if "search_ms_per_query" in tm:
        table(["Method", "Search latency (ms/query, top-10)"],
              [[method_label(m), f"{v:.3f}"] for m, v in tm["search_ms_per_query"].items()], [8, 6])
    para(f"Query text encoding: {tm.get('text_encode_ms_per_query', 0):.1f} ms/query (batched) on {tm.get('device')}.")
    doc.add_heading("9. Limitations and future work", 1)
    bullets([
        "200 queries give wide confidence intervals (one query = 0.5 R@1 points); results are reported with CIs and paired tests.",
        "Learned heads are trained on a few hundred clips with a frozen backbone; end-to-end fine-tuning on the full "
        "9k training split (as in CLIP4Clip) could change the picture.",
        "Audio and on-screen text (ASR/OCR) are not used although many MSR-VTT categories (music, news) rely on them.",
        "Future work: temporally-grounded benchmarks (e.g. order-sensitive caption sets), motion features (optical "
        "flow / VideoMAE), and hybrid scoring that adds temporal evidence only when the query contains temporal language.",
    ])
    doc.add_heading("10. Reproducibility", 1)
    bullets([
        "pip install -r requirements.txt  →  streamlit run app.py  →  Pipeline Runner → 'Run full pipeline'.",
        "All randomness is seeded (sampling seed, training seeds); outputs are cached under artifacts/.",
        f"Configuration used: {json.dumps({k: R['config'][k] for k in ('clip_model', 'num_frames', 'train_videos_per_category', 'epochs', 'train_seeds', 'seed')})}",
    ])
    doc.add_heading("Appendix A – Execution evidence (BITS Virtual Lab)", 1)
    for s in ["Pipeline Runner – full pipeline completed", "Text→Video search: 'a person is playing guitar on a stage' – Top-K playable clips with scores",
              "Text→Video search: edge case (typo / short / long query) with diagnostics", "Video→Text retrieval",
              "Evaluation page – metric tables and charts", "Research Investigation page – answer and significance tests"]:
        para(f"[Insert screenshot: {s}]")
    out.parent.mkdir(parents=True, exist_ok=True)
    doc.save(out)
    return out

#### `app.py`
**Streamlit dashboard** – pipeline runner, text→video search with playable clips, video→text, evaluation, research and report pages.

In [ ]:
%%writefile app.py
"""Streamlit dashboard - Natural-Language Video Retrieval on MSR-VTT.

Run:   streamlit run app.py
Everything (download, feature extraction, training, indexing, evaluation,
report generation) can be triggered from the "Pipeline Runner" page - no
manual script execution is needed.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st

from mvr.config import (ALL_SEARCH_METHODS, LEARNED_METHODS, METHOD_INFO, PARAM_FREE_METHODS, Config,
                        method_label)

st.set_page_config(page_title="MSR-VTT Text-to-Video Retrieval", page_icon="🎬", layout="wide")

st.markdown("""
<style>
.block-container {padding-top: 1.6rem; max-width: 1400px;}
.res-card {border: 1px solid rgba(128,128,128,.25); border-radius: 10px; padding: .55rem .7rem .3rem; margin-bottom: .6rem;}
.rank-badge {display:inline-block; background:#2a78d6; color:#fff; border-radius: 6px; padding: 0 .45rem; font-weight:600; margin-right:.35rem;}
.gt-badge {display:inline-block; background:#1baf7a; color:#fff; border-radius: 6px; padding: 0 .45rem; font-weight:600; margin-left:.35rem;}
.small-muted {color: rgba(128,128,128,1); font-size: .82rem;}
</style>
""", unsafe_allow_html=True)


# ---------------------------------------------------------------------------
# cached loaders
# ---------------------------------------------------------------------------
def get_cfg() -> Config:
    if "cfg" not in st.session_state:
        st.session_state.cfg = Config.load()
    return st.session_state.cfg


def _mtime(p: Path) -> float:
    return p.stat().st_mtime if p.exists() else 0.0


@st.cache_resource(show_spinner="Loading CLIP model...")
def cached_encoder(model: str, device: str):
    from mvr.encoders import ClipEncoder
    return ClipEncoder(model, device)


@st.cache_resource(show_spinner=False)
def cached_index(path: str, mtime: float):
    from mvr.index import VideoIndex
    return VideoIndex.load(Path(path))


@st.cache_data(show_spinner=False)
def cached_json(path: str, mtime: float):
    return json.loads(Path(path).read_text())


@st.cache_data(show_spinner=False)
def cached_features(path: str, mtime: float):
    d = np.load(path, allow_pickle=True)
    return {"query_emb": d["query_emb"], "index_frames": d["index_frames"]}


@st.cache_resource(show_spinner=False)
def cached_lexicon(path: str, mtime: float):
    from collections import Counter
    p = Path(path)
    return Counter(json.loads(p.read_text())) if p.exists() else Counter()


def state(cfg: Config) -> dict:
    return {
        "subset": cfg.subset_path.exists(),
        "features": cfg.features_path.exists(),
        "heads": any(cfg.models_dir.glob(f"*_T{cfg.num_frames}_seed*.pt")) if cfg.models_dir.exists() else False,
        "index": (cfg.index_dir / "meta.json").exists(),
        "results": cfg.results_path.exists(),
    }


def load_subset(cfg):
    return cached_json(str(cfg.subset_path), _mtime(cfg.subset_path)) if cfg.subset_path.exists() else None


def load_results(cfg):
    return cached_json(str(cfg.results_path), _mtime(cfg.results_path)) if cfg.results_path.exists() else None


def load_index(cfg):
    m = cfg.index_dir / "meta.json"
    return cached_index(str(cfg.index_dir), _mtime(m)) if m.exists() else None


def need(cfg, *keys) -> bool:
    s = state(cfg)
    missing = [k for k in keys if not s[k]]
    if missing:
        st.info("This page needs the pipeline outputs: **" + ", ".join(missing) + "**. "
                "Open **⚙️ Pipeline Runner** in the sidebar and click **Run full pipeline**.")
        return False
    return True


def fmt_metrics_df(table: dict, std: dict | None = None, cols=None) -> pd.DataFrame:
    cols = cols or ["R@1", "R@5", "R@10", "MRR", "mAP", "nDCG@10", "MdR", "MnR"]
    rows = []
    for m, d in table.items():
        r = {"Method": method_label(m)}
        for c in cols:
            v = d.get(c)
            s = (std or {}).get(m, {}).get(c, 0.0)
            if v is None:
                r[c] = ""
            elif c.startswith("R@"):
                r[c] = f"{v:.1f}" + (f" ± {s:.1f}" if s else "")
            elif c in ("MdR", "MnR"):
                r[c] = f"{v:.1f}" + (f" ± {s:.1f}" if s else "")
            else:
                r[c] = f"{v:.3f}" + (f" ± {s:.3f}" if s else "")
        rows.append(r)
    return pd.DataFrame(rows).set_index("Method")


FAMILY_COLORS = {"Parameter-free": "#2a78d6", "Learned head": "#eb6834", "External video model": "#1baf7a"}


def fam(m):
    return "Parameter-free" if m in PARAM_FREE_METHODS else ("Learned head" if m in LEARNED_METHODS else "External video model")


# ---------------------------------------------------------------------------
# sidebar
# ---------------------------------------------------------------------------
cfg = get_cfg()
PAGES = ["🏠 Overview", "⚙️ Pipeline Runner", "🔎 Text → Video Search", "🎞️ Video → Text",
         "📊 Evaluation", "🔬 Research Investigation", "📚 Dataset Browser", "📄 Report"]
with st.sidebar:
    st.markdown("### 🎬 MSR-VTT Video Retrieval")
    page = st.radio("Navigate", PAGES, key="page", label_visibility="collapsed")
    st.divider()
    s = state(cfg)
    st.caption("Pipeline status")
    for k, label in (("subset", "Data (100 clips / 200 queries)"), ("features", "CLIP frame features"),
                     ("heads", "Temporal heads"), ("index", "Vector index"), ("results", "Evaluation results")):
        st.markdown(("✅ " if s[k] else "⬜ ") + label)
    st.divider()
    st.caption(f"Model: `{cfg.clip_model}`  \nFrames/clip: {cfg.num_frames}  \nDevice: `{cfg.resolved_device()}`")


# ===========================================================================
# PAGE: Overview
# ===========================================================================
def page_overview():
    st.title("Text-to-Video Retrieval Using Semantic and Temporal Video Representations")
    st.markdown("**Dataset:** MSR-VTT · **Index:** 100 clips (5 × 20 categories) · **Queries:** 200 (2 captions/clip) "
                "· **Search corpus matrix:** 200 × 100")
    R = load_results(cfg)
    if R:
        t = R["t2v"]
        best = max(t, key=lambda m: t[m]["MRR"])
        c = st.columns(5)
        c[0].metric("Mean-pool R@1", f"{t['mean_pool']['R@1']:.1f}%")
        c[1].metric("Mean-pool R@5", f"{t['mean_pool']['R@5']:.1f}%")
        c[2].metric("Mean-pool MRR", f"{t['mean_pool']['MRR']:.3f}")
        c[3].metric("Best method (MRR)", method_label(best).split(" (")[0], f"{t[best]['MRR']-t['mean_pool']['MRR']:+.3f}")
        c[4].metric("Median rank (mean-pool)", f"{t['mean_pool']['MdR']:.0f}")
        st.success("**Research answer:** " + R["conclusions"]["answer"])
    else:
        st.info("No results yet - go to **⚙️ Pipeline Runner** and click **Run full pipeline**.")

    st.subheader("System architecture")
    st.graphviz_chart("""
digraph G { rankdir=LR; node [shape=box, style="rounded,filled", fontname="Helvetica", fontsize=10];
 subgraph cluster_off { label="Offline indexing"; style=dashed; color=gray;
   V [label="MSR-VTT clips\\n100 = 5 x 20 categories", fillcolor="#dbe8f8"];
   S [label="Frame sampling\\nT uniform segment mid-points", fillcolor="#dbe8f8"];
   E [label="CLIP ViT image encoder\\n(frozen) -> 512-d", fillcolor="#dbe8f8"];
   A [label="Temporal aggregation\\nmean | max | single | query-attn\\nSet-TFM | Temporal-TFM | LSTM | MLP", fillcolor="#dbe8f8"];
   I [label="FAISS IndexFlatIP\\n(L2-normalised vectors)", fillcolor="#d3f1e6"]; V->S->E->A->I; }
 subgraph cluster_tr { label="Training (disjoint train clips)"; style=dashed; color=gray;
   TR [label="Train clips + 20 captions\\nfrozen CLIP features", fillcolor="#fde3d8"];
   L [label="Symmetric InfoNCE\\n3 seeds, early stop on val", fillcolor="#fde3d8"]; TR->L; }
 L -> A [style=dashed, label="weights"];
 subgraph cluster_on { label="Online query"; style=dashed; color=gray;
   Q [label="Streamlit query box", fillcolor="#fbefcf"];
   P [label="Query processing\\nnormalise | spell-fix | templates | chunking", fillcolor="#fbefcf"];
   T [label="CLIP text encoder\\n(frozen) -> 512-d", fillcolor="#fbefcf"];
   K [label="Cosine similarity\\nTop-K + per-frame scores", fillcolor="#d3f1e6"];
   O [label="Playable clips + scores", fillcolor="#fbefcf"]; Q->P->T->K->O; }
 I -> K;
}""", use_container_width=True)

    st.subheader("Representations compared (Research Investigation 1)")
    rows = []
    for m in ALL_SEARCH_METHODS + ["xclip"]:
        rows.append({"Method": method_label(m), "Description": METHOD_INFO[m][1],
                     "Uses frame order": "yes" if m in ("temporal_tfm", "seq_lstm", "xclip") else "no",
                     "Trained": "yes" if m in LEARNED_METHODS else ("pre-trained only" if m == "xclip" else "no (zero-shot)")})
    st.dataframe(pd.DataFrame(rows), hide_index=True, use_container_width=True)


# ===========================================================================
# PAGE: Pipeline runner
# ===========================================================================
def page_runner():
    st.title("⚙️ Pipeline Runner")
    st.caption("Runs the complete pipeline from the front-end. Each step caches its output; re-running skips finished steps "
               "unless *Force* is ticked. Keep this page open while it runs.")
    with st.form("settings"):
        c1, c2, c3 = st.columns(3)
        models = ["openai/clip-vit-base-patch32", "openai/clip-vit-base-patch16", "openai/clip-vit-large-patch14"]
        cur = cfg.clip_model if cfg.clip_model in models else None
        clip_model = c1.selectbox("CLIP backbone", models + ([cfg.clip_model] if cur is None else []),
                                  index=(models + [cfg.clip_model]).index(cfg.clip_model))
        num_frames = c1.select_slider("Frames sampled per clip", [4, 8, 12, 16, 24, 32], value=cfg.num_frames)
        device = c1.selectbox("Device", ["auto", "cuda", "cpu"], index=["auto", "cuda", "cpu"].index(cfg.device))
        tv = c2.slider("Training clips per category (temporal heads)", 0, 150, cfg.train_videos_per_category, 5,
                       help="Disjoint from the 100 index clips. 0 = skip learned heads.")
        epochs = c2.slider("Training epochs", 5, 100, cfg.epochs, 5)
        n_seeds = c2.slider("Random seeds per head", 1, 5, len(cfg.train_seeds))
        cpu_fast = c3.checkbox("CPU-fast mode (if no GPU: ≤25 training clips/category, ≤20 epochs)", cfg.cpu_fast)
        use_x = c3.checkbox("Also evaluate X-CLIP (pretrained spatio-temporal video model)", cfg.use_xclip)
        seed = c3.number_input("Sampling seed", 0, 10_000, cfg.seed)
        local = c3.text_input("Offline mirror of the HF dataset repo (optional)", cfg.local_repo_dir,
                              help="Folder containing msrvtt_test_1k.json, msrvtt_train_9k.json, raw_data/, MSRVTT_Videos.zip")
        if st.form_submit_button("💾 Save settings"):
            cfg.clip_model, cfg.num_frames, cfg.device = clip_model, int(num_frames), device
            cfg.train_videos_per_category, cfg.epochs = int(tv), int(epochs)
            cfg.train_seeds = tuple(range(n_seeds))
            cfg.cpu_fast = cpu_fast
            cfg.use_xclip, cfg.seed, cfg.local_repo_dir = use_x, int(seed), local.strip()
            cfg.save()
            st.success("Settings saved.")

    force = st.checkbox("Force re-run (ignore cached outputs)")
    c = st.columns(7)
    run_all = c[0].button("▶️ Run full pipeline", type="primary", use_container_width=True)
    steps = {"1 · Data": c[1].button("1 · Data", use_container_width=True),
             "2 · Features": c[2].button("2 · Features", use_container_width=True),
             "3 · Train heads": c[3].button("3 · Train heads", use_container_width=True),
             "4 · Index": c[4].button("4 · Index", use_container_width=True),
             "5 · X-CLIP": c[5].button("5 · X-CLIP", use_container_width=True),
             "6 · Evaluate": c[6].button("6 · Evaluate", use_container_width=True)}
    chosen = [k for k, v in steps.items() if v]
    if run_all or chosen:
        from mvr import pipeline as P
        from mvr.data import prepare_subset
        bar = st.progress(0.0, text="Starting...")
        log = st.empty()
        lines: list[str] = []

        def prog(f, msg):
            bar.progress(float(min(max(f, 0), 1)), text=msg[:150])
            if not lines or lines[-1] != msg:
                lines.append(f"{time.strftime('%H:%M:%S')}  {msg}")
                log.code("\n".join(lines[-14:]))

        t0 = time.time()
        try:
            if run_all:
                P.run_all(cfg, prog, force)
            else:
                fns = {"1 · Data": lambda: prepare_subset(cfg, prog, force),
                       "2 · Features": lambda: P.step_features(cfg, prog, force),
                       "3 · Train heads": lambda: P.step_train(cfg, prog, force),
                       "4 · Index": lambda: P.step_index(cfg, prog),
                       "5 · X-CLIP": lambda: P.step_xclip(cfg, prog, force),
                       "6 · Evaluate": lambda: P.step_eval(cfg, prog)}
                for k in chosen:
                    fns[k]()
            st.cache_data.clear()
            bar.progress(1.0, text="Done")
            st.success(f"Finished in {time.time()-t0:.0f} s. Open **🔎 Text → Video Search** or **📊 Evaluation**.")
        except Exception as e:
            import traceback
            st.error(f"Pipeline failed: {e}")
            st.code(traceback.format_exc())

    st.divider()
    st.subheader("Cached outputs")
    files = [cfg.subset_path, cfg.features_path, cfg.models_dir, cfg.index_dir, cfg.results_path,
             Path(cfg.artifacts_dir) / "xclip_scores.npy"]
    st.dataframe(pd.DataFrame([{"Output": str(Path(f).relative_to(Path(cfg.artifacts_dir).parent)) if str(f).startswith(str(Path(cfg.artifacts_dir).parent)) else str(f),
                                "Exists": "✅" if Path(f).exists() else "—",
                                "Modified": time.strftime("%Y-%m-%d %H:%M", time.localtime(_mtime(Path(f)))) if Path(f).exists() else ""}
                               for f in files]), hide_index=True, use_container_width=True)
    xerr = Path(cfg.artifacts_dir) / "xclip_error.txt"
    if xerr.exists() and not (Path(cfg.artifacts_dir) / "xclip_scores.npy").exists():
        with st.expander("X-CLIP was skipped (optional) - details"):
            st.code(xerr.read_text()[-3000:])


# ===========================================================================
# PAGE: Text -> Video search
# ===========================================================================
EXAMPLES = ["a person is playing guitar on a stage", "a man is cooking food in a kitchen",
            "cars are racing on a track", "a cartoon character is talking", "a news anchor reports the news",
            "people are dancing at a party", "a dog is playing with a ball", "basketball",
            "a woman playng gutiar on stge",
            "a group of young people stand on a stage and sing a song while the crowd cheers, then the camera "
            "pans across the audience who are clapping and waving their hands in the air, and finally the lead "
            "singer bows and walks off the stage as the lights go down and the music slowly fades out"]


def _query_processor():
    from mvr.query import QueryProcessor
    enc = cached_encoder(cfg.clip_model, cfg.resolved_device())
    lexp = Path(cfg.data_dir) / "lexicon.json"
    return QueryProcessor(enc, cached_lexicon(str(lexp), _mtime(lexp)))


@st.cache_data(show_spinner=False)
def low_conf_threshold(index_mtime: float, method: str) -> float:
    """5th percentile of the ground-truth match score over the 200 eval queries."""
    idx = load_index(cfg)
    F = cached_features(str(cfg.features_path), _mtime(cfg.features_path))
    sub = load_subset(cfg)
    S = idx.scores(F["query_emb"], method)
    gt = np.array([q["video_idx"] for q in sub["queries"]])
    return float(np.percentile(S[np.arange(len(gt)), gt], 5))


def page_search():
    st.title("🔎 Text → Video Search")
    if not need(cfg, "subset", "features", "index"):
        return
    idx, sub = load_index(cfg), load_subset(cfg)
    eval_queries = [q["text"] for q in sub["queries"]]

    c1, c2 = st.columns([3, 1])
    with c2:
        ex = st.selectbox("Example queries", ["(type your own)"] + EXAMPLES + ["— from evaluation set —"] + eval_queries[:60])
    default = "" if ex.startswith("(") or ex.startswith("—") else ex
    with c1:
        with st.form("q"):
            query = st.text_input("Describe the video you are looking for", value=default,
                                  placeholder="e.g. A person is playing guitar on a stage.")
            a, b, c = st.columns([2, 1, 1])
            methods = [m for m in ALL_SEARCH_METHODS if m in idx.methods]
            method = a.selectbox("Video representation", methods, index=methods.index("mean_pool"),
                                 format_func=method_label)
            k = b.slider("Top-K", 1, 20, 5)
            compare = c.checkbox("Compare with mean-pool", value=False)
            o1, o2, o3 = st.columns(3)
            ac = o1.checkbox("Spelling correction", True)
            tp = o2.checkbox("Templates for short queries", True)
            ch = o3.checkbox("Chunk long queries", True)
            go = st.form_submit_button("Search", type="primary")
    if not go and not query:
        st.caption(f"**{method_label(method)}** – {METHOD_INFO[method][1]}")
        return

    from mvr.query import QueryError
    qp = _query_processor()
    try:
        t0 = time.perf_counter()
        pq = qp.process(query, auto_correct=ac, use_templates=tp, use_chunking=ch)
        q = qp.encode(pq)
        enc_ms = (time.perf_counter() - t0) * 1000
    except QueryError as e:
        st.error(str(e))
        return

    res = idx.search(q, method, k)
    thr = low_conf_threshold(_mtime(cfg.index_dir / "meta.json"), method)
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Query encoding", f"{enc_ms:.0f} ms")
    m2.metric("Index search", f"{res['latency_ms']:.2f} ms")
    m3.metric("Top-1 cosine score", f"{res['scores'][0]:.3f}")
    m4.metric("Query mode", pq.mode)
    with st.expander("Query processing diagnostics (edge-case handling)", expanded=bool(pq.warnings or pq.corrections or pq.mode != "normal")):
        st.write({"normalised": pq.normalized, "final text": pq.final_text, "BPE tokens": pq.n_tokens,
                  "texts encoded": pq.texts_to_encode, "steps": pq.steps or ["none"],
                  "unknown words": pq.unknown_words or ["none"]})
    for w in pq.warnings:
        st.warning(w)
    if res["scores"][0] < thr:
        st.warning(f"Low confidence: the best score ({res['scores'][0]:.3f}) is below the 5th percentile of correct-match "
                   f"scores on the evaluation set ({thr:.3f}). The collection may not contain a matching clip.")

    gt_vid = None
    if query.strip() in eval_queries:
        qi = eval_queries.index(query.strip())
        gt_vid = sub["queries"][qi]["video_idx"]
        S_all = idx.scores(q, method)[0]
        gt_rank = int(1 + (S_all > S_all[gt_vid]).sum())
        st.info(f"This is evaluation query #{qi}. Ground-truth clip **{sub['index'][gt_vid]['video_id']}** is ranked "
                f"**{gt_rank}** / {len(sub['index'])} by {method_label(method)}.")

    tbl = pd.DataFrame([{"Rank": r + 1, "Video": sub["index"][i]["video_id"],
                         "Category": sub["index"][i]["category_name"], "Cosine score": round(s, 4),
                         "Ground truth": "✅" if i == gt_vid else ""}
                        for r, (i, s) in enumerate(zip(res["idx"], res["scores"]))])
    st.dataframe(tbl, hide_index=True, use_container_width=True)

    st.subheader(f"Top-{k} clips · {method_label(method)}")
    ncol = 3
    for row in range(0, len(res["idx"]), ncol):
        cols = st.columns(ncol)
        for j, col in enumerate(cols):
            r = row + j
            if r >= len(res["idx"]):
                break
            vi, score = res["idx"][r], res["scores"][r]
            v = sub["index"][vi]
            with col:
                st.markdown(f"<div class='res-card'><span class='rank-badge'>#{r+1}</span>"
                            f"<b>{v['video_id']}</b> · {v['category_name']} · score <b>{score:.3f}</b>"
                            + ("<span class='gt-badge'>GT</span>" if vi == gt_vid else "") + "</div>",
                            unsafe_allow_html=True)
                vp = cfg.videos_dir / f"{v['video_id']}.mp4"
                if vp.exists():
                    st.video(str(vp))
                fs = res["frame_sims"][r]
                st.bar_chart(pd.DataFrame({"frame similarity": fs}, index=[f"f{t:02d}" for t in range(len(fs))]),
                             height=120, color="#2a78d6")
                with st.expander("Captions & frames"):
                    th = Path(cfg.thumbs_dir) / f"{v['video_id']}.jpg"
                    if th.exists():
                        st.image(str(th), use_container_width=True)
                    for cap in v["all_captions"][:5]:
                        st.markdown(f"- {cap}")

    if compare and method != "mean_pool":
        st.subheader("Mean-pool baseline for the same query")
        r2 = idx.search(q, "mean_pool", k)
        st.dataframe(pd.DataFrame([{"Rank": r + 1, "Video": sub["index"][i]["video_id"],
                                    "Category": sub["index"][i]["category_name"], "Cosine score": round(s, 4)}
                                   for r, (i, s) in enumerate(zip(r2["idx"], r2["scores"]))]),
                     hide_index=True, use_container_width=True)


# ===========================================================================
# PAGE: Video -> Text
# ===========================================================================
def page_v2t():
    st.title("🎞️ Video → Text Retrieval")
    st.caption("The reverse modality: given a clip, rank the 200 caption queries. Each clip has 2 relevant captions.")
    if not need(cfg, "subset", "features", "index"):
        return
    idx, sub = load_index(cfg), load_subset(cfg)
    F = cached_features(str(cfg.features_path), _mtime(cfg.features_path))
    opts = list(range(len(sub["index"])))
    c1, c2, c3 = st.columns([2, 2, 1])
    vi = c1.selectbox("Clip", opts, format_func=lambda i: f"{sub['index'][i]['video_id']} · {sub['index'][i]['category_name']}")
    methods = [m for m in ALL_SEARCH_METHODS if m in idx.methods]
    method = c2.selectbox("Video representation", methods, index=methods.index("mean_pool"), format_func=method_label)
    k = c3.slider("Top-K captions", 1, 20, 10)
    S = idx.scores(F["query_emb"], method)[:, vi]
    order = np.argsort(-S)[:k]
    left, right = st.columns([1, 1])
    with left:
        st.video(str(cfg.videos_dir / f"{sub['index'][vi]['video_id']}.mp4"))
        th = Path(cfg.thumbs_dir) / f"{sub['index'][vi]['video_id']}.jpg"
        if th.exists():
            st.image(str(th), caption="Sampled frames", use_container_width=True)
    with right:
        st.dataframe(pd.DataFrame([{"Rank": r + 1, "Caption": sub["queries"][qi]["text"], "Score": round(float(S[qi]), 4),
                                    "Relevant": "✅" if sub["queries"][qi]["video_idx"] == vi else ""}
                                   for r, qi in enumerate(order)]), hide_index=True, use_container_width=True)


# ===========================================================================
# PAGE: Evaluation
# ===========================================================================
def page_eval():
    import altair as alt
    st.title("📊 Quantitative Performance Evaluation")
    if not need(cfg, "results"):
        return
    R = load_results(cfg)
    st.caption(f"{R['n_queries']} queries × {R['n_videos']} clips · backbone `{R['config']['clip_model']}` · "
               f"{R['config']['num_frames']} frames/clip · learned heads: mean ± sd over seeds {R['config']['train_seeds']} · "
               f"generated {R['generated_at']}")
    t1, t2, t3, t4, t5, t6 = st.tabs(["Text→Video", "Video→Text", "Charts", "Frames & per-category",
                                      "Edge-case robustness", "Efficiency"])
    with t1:
        st.dataframe(fmt_metrics_df(R["t2v"], R["t2v_std"]), use_container_width=True)
        st.caption("R@K in %, higher is better · MdR/MnR = median/mean rank of the correct clip, lower is better · "
                   "with one relevant clip per query, mAP = MRR.")
    with t2:
        st.dataframe(fmt_metrics_df(R["v2t"], R["v2t_std"]), use_container_width=True)
        st.caption("Video→Text: 2 relevant captions per clip; R@K = at least one relevant caption in top-K; mAP uses both.")
    with t3:
        rows = []
        for m, d in R["t2v"].items():
            for kk in ("R@1", "R@5", "R@10"):
                rows.append({"Method": method_label(m), "Metric": kk, "Value": d[kk], "Family": fam(m)})
        df = pd.DataFrame(rows)
        order = [method_label(m) for m in R["t2v"]]
        mets = ["R@1", "R@5", "R@10"]
        ch = alt.Chart(df).mark_bar(cornerRadiusTopRight=3, cornerRadiusBottomRight=3).encode(
            y=alt.Y("Method:N", sort=order, title=None, axis=alt.Axis(labelLimit=260, labelOverlap=False)),
            x=alt.X("Value:Q", title="Recall (%)"),
            yOffset=alt.YOffset("Metric:N", sort=mets),
            color=alt.Color("Metric:N", sort=mets, scale=alt.Scale(domain=mets, range=["#2a78d6", "#eb6834", "#1baf7a"])),
            tooltip=["Method", "Metric", alt.Tooltip("Value:Q", format=".1f")]).properties(height=60 * len(order),
                                                                                           title="Text→Video Recall@K")
        st.altair_chart(ch, use_container_width=True)
        mdf = pd.DataFrame([{"Method": method_label(m), "MRR": d["MRR"], "sd": R["t2v_std"].get(m, {}).get("MRR", 0),
                             "Family": fam(m)} for m, d in R["t2v"].items()])
        mdf["lo"], mdf["hi"] = mdf.MRR - mdf.sd, mdf.MRR + mdf.sd
        base = alt.Chart(mdf).encode(y=alt.Y("Method:N", sort=order, title=None, axis=alt.Axis(labelLimit=260, labelOverlap=False)))
        bars = base.mark_bar(cornerRadiusTopRight=3, cornerRadiusBottomRight=3).encode(
            x=alt.X("MRR:Q", title="MRR"),
            color=alt.Color("Family:N", scale=alt.Scale(domain=list(FAMILY_COLORS), range=list(FAMILY_COLORS.values()))),
            tooltip=["Method", alt.Tooltip("MRR:Q", format=".3f"), alt.Tooltip("sd:Q", format=".3f")])
        err = base.mark_errorbar(ticks=True).encode(x=alt.X("lo:Q", title="MRR"), x2="hi:Q")
        rule = alt.Chart(pd.DataFrame({"x": [R["t2v"]["mean_pool"]["MRR"]]})).mark_rule(strokeDash=[4, 4], color="gray").encode(x="x:Q")
        st.altair_chart((bars + err + rule).properties(height=44 * len(order), title="MRR ± sd (dashed = mean-pool baseline)"),
                        use_container_width=True)
    with t4:
        sw = R["frame_sweep"]
        df = pd.DataFrame([{"Method": method_label(m), "Frames": int(k), "MRR": v["MRR"], "R@1": v["R@1"]}
                           for m in sw for k, v in sw[m].items()])
        sel = st.multiselect("Methods", sorted(df.Method.unique()),
                             default=[method_label(m) for m in ("mean_pool", "max_pool", "query_attn", "temporal_tfm") if m in sw])
        ch = alt.Chart(df[df.Method.isin(sel)]).mark_line(point=alt.OverlayMarkDef(size=60), strokeWidth=2).encode(
            x=alt.X("Frames:Q", scale=alt.Scale(type="log", base=2)), y="MRR:Q", color="Method:N",
            tooltip=["Method", "Frames", alt.Tooltip("MRR:Q", format=".3f"), alt.Tooltip("R@1:Q", format=".1f")]
        ).properties(height=320, title="MRR vs frames sampled per clip")
        st.altair_chart(ch, use_container_width=True)
        pc = R["per_category"]
        metric = st.radio("Per-category metric", ["R@5", "MRR"], horizontal=True)
        hd = pd.DataFrame([{"Category": c, "Method": method_label(m), metric: v[metric]} for m in pc for c, v in pc[m].items()])
        hm = alt.Chart(hd).mark_rect().encode(
            x=alt.X("Method:N", sort=[method_label(m) for m in pc], axis=alt.Axis(labelAngle=-35, labelLimit=160)),
            y="Category:N", color=alt.Color(f"{metric}:Q", scale=alt.Scale(scheme="blues")),
            tooltip=["Category", "Method", alt.Tooltip(f"{metric}:Q", format=".3f")]).properties(height=520)
        st.altair_chart(hm, use_container_width=True)
    with t5:
        rb = R.get("robustness", {})
        if "rows" in rb:
            st.dataframe(pd.DataFrame([{"Query condition": r["condition"], "Handling": r["handling"],
                                        "R@1": round(r["R@1"], 1), "R@5": round(r["R@5"], 1), "MRR": round(r["MRR"], 3),
                                        "MdR": r["MdR"]} for r in rb["rows"]]), hide_index=True, use_container_width=True)
            st.caption(f"Mean-pool index. Typo queries containing unknown words: {rb['n_typo_queries_with_unknown_words']} / 200; "
                       f"detailed queries exceeding 77 tokens: {rb['n_long_queries']} / 200.")
            with st.expander("Examples of perturbed queries"):
                st.json(rb["examples"])
        else:
            st.warning(f"Robustness study unavailable: {rb.get('error')}")
    with t6:
        tm = R.get("timing", {})
        c = st.columns(4)
        c[0].metric("Decode / clip", f"{tm.get('decode_s_per_video', 0)*1000:.0f} ms")
        c[1].metric("CLIP encode / clip", f"{tm.get('encode_s_per_video', 0)*1000:.0f} ms")
        c[2].metric("Text encode / query", f"{tm.get('text_encode_ms_per_query', 0):.1f} ms")
        c[3].metric("Index build", f"{tm.get('build_ms', 0):.1f} ms")
        if "search_ms_per_query" in tm:
            st.dataframe(pd.DataFrame([{"Method": method_label(m), "Search latency (ms/query, top-10)": round(v, 3)}
                                       for m, v in tm["search_ms_per_query"].items()]), hide_index=True)
        st.caption(f"Device: {tm.get('device')} · index backend: {tm.get('backend', 'n/a')}")


# ===========================================================================
# PAGE: Research investigation
# ===========================================================================
def page_research():
    import altair as alt
    st.title("🔬 Research Investigation 1")
    st.markdown("> *Is mean-pooling frame embeddings sufficient, or does explicitly modelling temporal relationships "
                "provide superior retrieval performance?*")
    if not need(cfg, "results"):
        return
    R = load_results(cfg)
    C = R["conclusions"]
    st.subheader("Answer")
    {"temporal_better": st.success, "adaptation_not_order": st.info, "mean_pool_sufficient": st.info,
     "mixed": st.warning}.get(C["verdict"], st.info)(C["answer"])
    st.subheader("Evidence")
    for p in C["points"]:
        st.markdown(f"- {p}")

    st.subheader("Experimental design (controlled ablation ladder)")
    st.markdown("""
| Step | Representation | What it adds | Isolates |
|---|---|---|---|
| 0 | Single middle frame | – | value of using many frames |
| 1 | **Mean pooling** (baseline) | average of all frames | – |
| 1b | Max / query-attentive pooling | non-uniform weighting (zero-shot) | smarter order-free pooling |
| 2 | MLP adapter (trained) | in-domain fine-tuning, **no** frame interaction | domain adaptation |
| 3 | Set Transformer (trained) | frame–frame attention, **no** order | inter-frame interaction |
| 4 | Temporal Transformer (trained) | + temporal position embeddings | **temporal order** |
| 4b | Temporal LSTM (trained) | recurrent ordered model | temporal order (alt. architecture) |
| ext | X-CLIP (pre-trained) | native spatio-temporal video encoder | different model family |

All learned heads share the frozen CLIP features, data (disjoint MSR-VTT training clips), loss (symmetric InfoNCE),
optimiser and early-stopping rule, are zero-initialised residuals of mean pooling, and are trained with 3 seeds.
Significance: paired bootstrap (10k resamples) 95% CIs on ΔMRR/ΔR@K and Wilcoxon signed-rank tests on per-query
reciprocal ranks with Holm–Bonferroni correction.""")

    st.subheader("Paired significance tests")
    rows = []
    for r in R["significance"]:
        rows.append({"Comparison": f"{method_label(r['method'])}  vs  {method_label(r['reference'])}",
                     "ΔMRR": f"{r['MRR']['delta']:+.3f}", "95% CI (MRR)": f"[{r['MRR']['ci_low']:+.3f}, {r['MRR']['ci_high']:+.3f}]",
                     "ΔR@1 (pts)": f"{r['R@1']['delta']:+.1f}", "ΔR@5 (pts)": f"{r['R@5']['delta']:+.1f}",
                     "Wilcoxon p": f"{r['wilcoxon_p']:.3g}", "Holm p": f"{r['wilcoxon_p_holm']:.3g}",
                     "better / worse / tie": f"{r['n_better']} / {r['n_worse']} / {r['n_tie']}"})
    st.dataframe(pd.DataFrame(rows), hide_index=True, use_container_width=True)

    c1, c2 = st.columns(2)
    with c1:
        oa = R["order_ablation"]
        df = pd.DataFrame([{"Method": method_label(m), "Frame order": c, "MRR": oa[m][c]["MRR"]}
                           for m in oa for c in ("original", "shuffled", "reversed")])
        conds = ["original", "shuffled", "reversed"]
        ch = alt.Chart(df).mark_bar().encode(
            y=alt.Y("Method:N", sort=[method_label(m) for m in oa], axis=alt.Axis(labelLimit=220, labelOverlap=False), title=None),
            yOffset=alt.YOffset("Frame order:N", sort=conds), x=alt.X("MRR:Q", scale=alt.Scale(zero=False)),
            color=alt.Color("Frame order:N", sort=conds, scale=alt.Scale(domain=conds, range=["#2a78d6", "#eb6834", "#1baf7a"])),
            tooltip=["Method", "Frame order", alt.Tooltip("MRR:Q", format=".4f")]).properties(
            height=48 * len(oa), title="Test-time temporal-order ablation")
        st.altair_chart(ch, use_container_width=True)
    with c2:
        tl = R["temporal_language"]
        st.markdown(f"**Queries with temporal language:** {tl['n_temporal']} of {tl['n_temporal'] + tl['n_other']}")
        st.dataframe(pd.DataFrame([{"Method": method_label(m), "MRR (temporal-language queries)": v["temporal_MRR"],
                                    "MRR (other queries)": v["other_MRR"]} for m, v in tl["by_method"].items()]),
                     hide_index=True, use_container_width=True)
        if tl["examples"]:
            with st.expander("Examples"):
                for e in tl["examples"]:
                    st.markdown(f"- {e}")
    pc = R.get("positive_control", {}).get("mrr")
    if pc:
        st.subheader("Positive control – synthetic task where only frame order matters")
        st.caption("Clips show state A then B (or B then A); captions encode the ordered pair. Confirms that the "
                   "temporal heads/evaluation CAN detect a temporal benefit when it exists.")
        st.dataframe(pd.DataFrame([{"Method": method_label(m), "MRR": round(v, 3)} for m, v in pc.items()]),
                     hide_index=True)
    tr = R.get("training", {})
    if tr:
        st.subheader("Training curves (validation MRR on held-out TRAIN clips)")
        df = pd.DataFrame([{"Run": k, "Head": method_label(k.rsplit("_seed", 1)[0]), "Epoch": e, "Val MRR": v}
                           for k, h in tr.items() for e, v in zip(h["epoch"], h["val_mrr"])])
        ch = alt.Chart(df).mark_line(strokeWidth=1.6).encode(x="Epoch:Q", y=alt.Y("Val MRR:Q", scale=alt.Scale(zero=False)),
                                                             color=alt.Color("Head:N", scale=alt.Scale(range=["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])), detail="Run:N",
                                                             tooltip=["Run", "Epoch", alt.Tooltip("Val MRR:Q", format=".3f")])
        st.altair_chart(ch.properties(height=300), use_container_width=True)
        st.dataframe(pd.DataFrame([{"Run": k, "Best epoch": h["best_epoch"], "Best val MRR": round(h["best_val_mrr"], 3),
                                    "Params": h["n_params"], "Train clips": h["n_train_videos"], "Seconds": h["train_seconds"]}
                                   for k, h in tr.items()]), hide_index=True, use_container_width=True)
    if "tau_selection" in R:
        st.caption(f"Query-attentive pooling temperature τ = {R['tau']} selected on the training pool: "
                   + ", ".join(f"τ={k}: MRR {v:.3f}" for k, v in R["tau_selection"]["grid_mrr"].items()))


# ===========================================================================
# PAGE: Dataset browser
# ===========================================================================
def page_dataset():
    st.title("📚 Dataset Browser")
    if not need(cfg, "subset"):
        return
    sub = load_subset(cfg)
    p = sub["protocol"]
    c = st.columns(4)
    c[0].metric("Index clips", p["n_index"])
    c[1].metric("Queries", p["n_queries"])
    c[2].metric("Categories", len(sub["category_names"]))
    c[3].metric("Training clips (disjoint)", p["n_train"])
    t1, t2 = st.tabs(["Clips", "Queries (200)"])
    with t1:
        cat = st.selectbox("Category", sub["category_names"])
        vids = [v for v in sub["index"] if v["category_name"] == cat]
        cols = st.columns(len(vids) if vids else 1)
        for col, v in zip(cols, vids):
            with col:
                st.video(str(cfg.videos_dir / f"{v['video_id']}.mp4"))
                st.markdown(f"**{v['video_id']}** · {v.get('duration', 0)} s · {v.get('width', '?')}×{v.get('height', '?')}")
                for qq in v["queries"]:
                    st.caption("🔍 " + qq)
    with t2:
        f = st.text_input("Filter queries")
        df = pd.DataFrame([{"qid": q["qid"], "Query": q["text"], "Video": q["video_id"],
                            "Category": sub["category_names"][q["category"]]} for q in sub["queries"]])
        if f:
            df = df[df.Query.str.contains(f, case=False)]
        st.dataframe(df, hide_index=True, use_container_width=True, height=520)


# ===========================================================================
# PAGE: Report
# ===========================================================================
def page_report():
    st.title("📄 Technical Report")
    if not need(cfg, "results"):
        return
    out = Path(cfg.artifacts_dir) / "MIR_Assignment2_Technical_Report.docx"
    if st.button("Generate / refresh report (.docx)", type="primary"):
        from mvr.report import build_report
        with st.spinner("Building report..."):
            build_report(cfg, out)
        st.success("Report generated.")
    if out.exists():
        st.download_button("⬇️ Download report", out.read_bytes(), file_name=out.name,
                           mime="application/vnd.openxmlformats-officedocument.wordprocessingml.document")
    figs = sorted(Path(cfg.figures_dir).glob("*.png")) if Path(cfg.figures_dir).exists() else []
    for f in figs:
        st.image(str(f), caption=f.stem, use_container_width=True)


{"🏠 Overview": page_overview, "⚙️ Pipeline Runner": page_runner, "🔎 Text → Video Search": page_search,
 "🎞️ Video → Text": page_v2t, "📊 Evaluation": page_eval, "🔬 Research Investigation": page_research,
 "📚 Dataset Browser": page_dataset, "📄 Report": page_report}[page]()

## 2. Configuration

In [ ]:
# Import the freshly written package and load the configuration (dashboard-saved settings are honoured).
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image, Video, HTML
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

from mvr.config import Config, method_label, METHOD_INFO
from mvr import pipeline as P
from mvr.data import prepare_subset

cfg = Config.load()
print("device:", cfg.resolved_device(), "| effective training clips/category:", cfg.eff_train_per_cat(),
      "| effective epochs:", cfg.eff_epochs())
pd.DataFrame(cfg.to_dict().items(), columns=["setting", "value"])

In [ ]:
# Compact progress printer for long-running steps (prints at most every 10% to keep outputs readable).
def progress_printer(label):
    state = {"last": -1.0}
    def cb(frac, msg):
        if frac >= 1.0 or frac - state["last"] >= 0.10:
            state["last"] = frac
            print(f"[{label} {frac*100:5.1f}%] {msg}", flush=True)
    return cb

## 3. Data ingestion
* **Index:** 5 clips from each of the 20 MSR-VTT categories, drawn from the official 1k-A *test* split (seeded, stratified).
  Every candidate clip is decoded before acceptance; corrupt clips are replaced automatically.
* **Queries:** 2 human captions per clip (the official 1k-A caption + one other) → 200 queries.
* **Training pool:** clips from the 9k *train* split, **disjoint** from the index (asserted in code), used only by the learned temporal heads.
* **Content filter:** MSR-VTT is crowd-captioned web video; clips whose captions use sexual/explicit wording are excluded
  from both the index and the training pool (`is_sensitive()` in `mvr/data.py`), and replaced by the next sampled clip.

In [ ]:
t0 = time.time()
subset = prepare_subset(cfg, progress_printer("data"))
print(f"done in {time.time()-t0:.1f}s")
pd.DataFrame([subset["protocol"]])

In [ ]:
# Index composition: 5 clips per category, with clip metadata from the decoder.
idx_df = pd.DataFrame([{"video_id": v["video_id"], "category": v["category_name"], "duration_s": v.get("duration"),
                        "fps": round(v.get("fps", 0), 1), "frames": v.get("n_frames"),
                        "resolution": f"{v.get('width')}x{v.get('height')}"} for v in subset["index"]])
display(idx_df.groupby("category").size().rename("clips").to_frame().T)
idx_df.describe(include="all").loc[["count", "mean", "min", "max"], ["duration_s", "fps", "frames"]]

In [ ]:
# Sample of the 200 queries (2 per clip).
pd.DataFrame(subset["queries"])[["qid", "text", "video_id"]].head(10)

In [ ]:
# Raw frames sampled from two clips (TSN-style segment mid-points) and one embedded, playable clip.
for v in subset["index"][:2]:
    print(v["video_id"], "-", v["category_name"], "-", v["queries"][0])
    display(Image(filename=str(Path(cfg.thumbs_dir) / f"{v['video_id']}.jpg")))
v0 = subset["index"][0]
Video(str(cfg.videos_dir / f"{v0['video_id']}.mp4"), embed=True, width=360)

## 4. Visual feature extraction & cross-modal embedding
Each clip → **T = 16** uniformly spaced frames → frozen **CLIP ViT** image tower + projection → 512-d L2-normalised vectors.
Queries → CLIP text Transformer + projection → the **same** 512-d space, so cosine similarity = dot product.

In [ ]:
t0 = time.time()
F = P.step_features(cfg, progress_printer("features"))
print(f"done in {time.time()-t0:.1f}s")
for k in ("index_frames", "query_emb", "train_frames", "train_text"):
    print(f"{k:13s} {F[k].shape}  dtype={F[k].dtype}")
json.loads(str(F["timing"]))

In [ ]:
# Sanity check of the shared space: matching (query, clip) pairs must be far more similar than random pairs.
fr, Q = F["index_frames"], F["query_emb"]
gt = np.array([q["video_idx"] for q in subset["queries"]])
from mvr.temporal import pool_video
V = pool_video(fr, "mean_pool")
S = Q @ V.T
pos = S[np.arange(len(gt)), gt]
neg = S[np.ones_like(S, bool) & (np.arange(S.shape[1])[None, :] != gt[:, None])]
print(f"cosine(query, matching clip)  mean={pos.mean():.3f} sd={pos.std():.3f}")
print(f"cosine(query, other clips)    mean={neg.mean():.3f} sd={neg.std():.3f}")
print(f"norms of frame embeddings ~ {np.linalg.norm(fr, axis=-1).mean():.3f} (L2-normalised)")

## 5. Learned temporal heads (trained on the disjoint training pool)
Four heads share the frozen CLIP features, data, loss (symmetric InfoNCE), optimiser, augmentation and early stopping:
**MLP adapter** (control, no frame interaction), **Set Transformer** (frame interaction, no order),
**Temporal Transformer** (+ temporal position embeddings) and **LSTM** (ordered recurrence).
All are zero-initialised residuals of mean pooling and trained with 3 seeds. The query-attention temperature τ is chosen on
the training pool (never on the test queries).

In [ ]:
t0 = time.time()
hists = P.step_train(cfg, progress_printer("train"))
print(f"done in {time.time()-t0:.1f}s")
pd.DataFrame([{"run": k, "params": h["n_params"], "train clips": h["n_train_videos"], "val clips": h["n_val_videos"],
               "best epoch": h["best_epoch"], "best val MRR": round(h["best_val_mrr"], 4), "seconds": h["train_seconds"]}
              for k, h in hists.items()])

In [ ]:
json.loads((cfg.models_dir / "tau.json").read_text())

## 6. Vector index (FAISS)

In [ ]:
index = P.step_index(cfg, progress_printer("index"))
print("backend:", index.backend, "| representations:", index.methods)
print("index matrix per representation:", {m: M.shape for m, M in index.pooled.items()})

## 7. External reference: X-CLIP (pre-trained spatio-temporal video model)
Optional; a failure here does not affect the rest of the pipeline.

In [ ]:
print("X-CLIP:", P.step_xclip(cfg, progress_printer("xclip")))

## 8. Quantitative evaluation
Metrics: **R@1/5/10** (%), **MRR**, **mAP**, **nDCG@10**, **MdR** / **MnR** (median / mean rank of the correct item).
Text→Video has one relevant clip per query (so mAP = MRR); Video→Text has two relevant captions per clip.
Learned heads are reported as mean ± sd over 3 seeds.

In [ ]:
t0 = time.time()
R = P.step_eval(cfg, progress_printer("eval"))
print(f"done in {time.time()-t0:.1f}s")

def metric_table(tab, std):
    cols = ["R@1", "R@5", "R@10", "MRR", "mAP", "nDCG@10", "MdR", "MnR"]
    rows = []
    for m, d in tab.items():
        r = {"Method": method_label(m)}
        for c in cols:
            s = std.get(m, {}).get(c, 0)
            f = "{:.1f}" if (c.startswith("R@") or c in ("MdR", "MnR")) else "{:.3f}"
            r[c] = f.format(d[c]) + ((" ± " + f.format(s)) if s else "")
        rows.append(r)
    return pd.DataFrame(rows).set_index("Method")

display(Markdown("**Table 1 – Text → Video (200 queries × 100 clips)**"))
display(metric_table(R["t2v"], R["t2v_std"]))
display(Markdown("**Table 2 – Video → Text (100 clips × 200 captions)**"))
display(metric_table(R["v2t"], R["v2t_std"]))

In [ ]:
Image(filename=str(cfg.figures_dir / "main_bars.png"))

## 9. Research Investigation 1 – is mean pooling sufficient?
**Design (ablation ladder):** single frame → **mean pooling** → zero-shot max / query-attentive pooling → MLP adapter
(domain-adaptation control) → Set Transformer (interaction, no order) → Temporal Transformer / LSTM (explicit order),
plus X-CLIP. **Tests:** paired bootstrap 95% CIs, Wilcoxon signed-rank on per-query reciprocal ranks, Holm correction;
test-time frame shuffling/reversal; frame-count sweep; temporal-language query subset; synthetic positive control.

In [ ]:
# Paired significance tests (A vs B; positive delta = A better).
pd.DataFrame([{"A": method_label(r["method"]), "B": method_label(r["reference"]),
               "ΔMRR": round(r["MRR"]["delta"], 4),
               "95% CI": f"[{r['MRR']['ci_low']:+.3f}, {r['MRR']['ci_high']:+.3f}]",
               "ΔR@1": round(r["R@1"]["delta"], 1), "ΔR@5": round(r["R@5"]["delta"], 1),
               "Wilcoxon p": f"{r['wilcoxon_p']:.3g}", "Holm p": f"{r['wilcoxon_p_holm']:.3g}",
               "better/worse/tie": f"{r['n_better']}/{r['n_worse']}/{r['n_tie']}"} for r in R["significance"]])

In [ ]:
# Test-time temporal-order ablation: order-agnostic methods must be unchanged; order-aware ones reveal order use.
oa = R["order_ablation"]
display(pd.DataFrame({method_label(m): {c: round(oa[m][c]["MRR"], 4) for c in ("original", "shuffled", "reversed")}
                      for m in oa}).T)
Image(filename=str(cfg.figures_dir / "order.png"))

In [ ]:
# Number of sampled frames vs MRR.
sw = R["frame_sweep"]
display(pd.DataFrame({method_label(m): {int(k): round(v["MRR"], 4) for k, v in sw[m].items()} for m in sw}).T)
Image(filename=str(cfg.figures_dir / "frame_sweep.png"))

In [ ]:
# Queries with explicit temporal language vs the rest.
tl = R["temporal_language"]
print(f"{tl['n_temporal']} temporal-language queries, e.g.:", tl["examples"][:3])
pd.DataFrame({method_label(m): v for m, v in tl["by_method"].items()}).T.round(4)

In [ ]:
Image(filename=str(cfg.figures_dir / "per_category.png"))

In [ ]:
# Positive control: synthetic clips where ONLY frame order distinguishes them.
pd.DataFrame({method_label(m): {"MRR (order-only task)": round(v, 3)} for m, v in R["positive_control"]["mrr"].items()}).T

In [ ]:
Image(filename=str(cfg.figures_dir / "training.png"))

In [ ]:
# Data-driven conclusions (generated from the numbers above).
C = R["conclusions"]
display(Markdown("### Answer\n> " + C["answer"]))
display(Markdown("### Evidence\n" + "\n".join("- " + p for p in C["points"])))

## 10. Query edge-case handling
The engine handles: empty / punctuation-only input, unicode noise and URLs, misspelled / unknown vocabulary
(frequency-aware edit-distance correction), very short queries (prompt-template ensemble), very long queries
(> 77 CLIP tokens → sentence-aware chunking + averaging), and non-English text (warning).

In [ ]:
from mvr.query import QueryProcessor, QueryError
enc = P.get_encoder(cfg)
qp = QueryProcessor(enc, P.load_lexicon(cfg))
edge_queries = ["", "  ?!...  ", "a woman playng gutiar on stge", "basketball",
                "A  man\tis   cooking   in a kitchen 😀 https://example.com",
                "un hombre toca la guitarra en el escenario",
                "a group of young people stand on a stage and sing a song while the crowd cheers, then the camera pans "
                "across the audience who are clapping and waving their hands in the air, and finally the lead singer "
                "bows and walks off the stage as the lights go down and the music slowly fades out"]
rows = []
for q in edge_queries:
    try:
        pq = qp.process(q)
        rows.append({"input": q[:60], "mode": pq.mode, "tokens": pq.n_tokens, "final text": pq.final_text[:60],
                     "#texts encoded": len(pq.texts_to_encode), "steps": "; ".join(pq.steps)[:90],
                     "warnings": "; ".join(pq.warnings)[:90]})
    except QueryError as e:
        rows.append({"input": repr(q), "mode": "rejected", "warnings": str(e)[:90]})
pd.DataFrame(rows)

In [ ]:
# Measured effect of the edge-case handling on all 200 (perturbed) queries, mean-pool index.
rb = R["robustness"]
display(pd.DataFrame(rb["rows"])[["condition", "handling", "R@1", "R@5", "MRR", "MdR"]].round(3))
print("examples:", json.dumps(rb["examples"], indent=1)[:600])

## 11. Live text → video retrieval (Top-K with scores)
`search()` runs the full online path: query processing → CLIP text embedding → cosine similarity against the index →
Top-K clips with scores, category, per-frame similarity and the ground truth (when the query is from the evaluation set).

In [ ]:
eval_texts = [q["text"] for q in subset["queries"]]

def search(query, method="mean_pool", k=5, play_top=True):
    pq = qp.process(query)
    t0 = time.perf_counter(); qv = qp.encode(pq); enc_ms = (time.perf_counter() - t0) * 1000
    res = index.search(qv, method, k)
    gt_idx = subset["queries"][eval_texts.index(query)]["video_idx"] if query in eval_texts else None
    display(Markdown(f"**Query:** `{query[:120]}` → *{pq.final_text[:120]}*  |  representation: **{method_label(method)}**  |  "
                     f"encode {enc_ms:.0f} ms, search {res['latency_ms']:.2f} ms"))
    display(pd.DataFrame([{"rank": r + 1, "video": subset["index"][i]["video_id"],
                           "category": subset["index"][i]["category_name"], "cosine score": round(s, 4),
                           "peak frame": int(np.argmax(res["frame_sims"][r])),
                           "ground truth": "✔" if i == gt_idx else "",
                           "a caption of this clip": subset["index"][i]["all_captions"][0][:70]}
                          for r, (i, s) in enumerate(zip(res["idx"], res["scores"]))]))
    top = subset["index"][res["idx"][0]]
    display(Image(filename=str(Path(cfg.thumbs_dir) / f"{top['video_id']}.jpg")))
    if play_top:
        display(Video(str(cfg.videos_dir / f"{top['video_id']}.mp4"), embed=True, width=360))

search("a person is playing guitar on a stage")

In [ ]:
search("a woman playng gutiar on stge", play_top=False)       # typo-corrected query

In [ ]:
search(subset["queries"][10]["text"], method="temporal_tfm", play_top=False)   # evaluation query, temporal model

In [ ]:
search("basketball", method="query_attn", play_top=False)     # very short query -> template ensemble

## 12. Video → text retrieval (the reverse modality)

In [ ]:
def video_to_text(vi=0, method="mean_pool", k=5):
    S = index.scores(F["query_emb"], method)[:, vi]
    order = np.argsort(-S)[:k]
    v = subset["index"][vi]
    display(Markdown(f"**Clip:** {v['video_id']} ({v['category_name']}) | representation: **{method_label(method)}**"))
    display(Image(filename=str(Path(cfg.thumbs_dir) / f"{v['video_id']}.jpg")))
    display(pd.DataFrame([{"rank": r + 1, "caption": subset["queries"][qi]["text"], "score": round(float(S[qi]), 4),
                           "relevant": "✔" if subset["queries"][qi]["video_idx"] == vi else ""}
                          for r, qi in enumerate(order)]))

video_to_text(0)
video_to_text(42, method="temporal_tfm")

## 13. Efficiency

In [ ]:
tm = R["timing"]
display(pd.DataFrame([{k: (round(v, 4) if isinstance(v, float) else v) for k, v in tm.items() if k != "search_ms_per_query"}]).T
        .rename(columns={0: "value"}))
pd.DataFrame([{"representation": method_label(m), "search ms/query (top-10)": round(v, 4)}
              for m, v in tm.get("search_ms_per_query", {}).items()])

## 14. Technical report (.docx) generated from these results

In [ ]:
from mvr.report import build_report
report_path = build_report(cfg, Path(cfg.artifacts_dir) / "MIR_Assignment2_Technical_Report.docx")
print("report written to:", report_path.resolve())
Image(filename=str(cfg.figures_dir / "architecture.png"))

## 15. Streamlit interactive dashboard
The dashboard is launched **from this notebook** (no manual script execution). Open the printed URL in the lab browser:
type any natural-language query, choose the video representation and Top-K, and play the retrieved clips with their
similarity scores. Other pages: Pipeline Runner, Video → Text, Evaluation, Research Investigation, Dataset Browser, Report.

In [ ]:
import subprocess, urllib.request
PORT = 8501
def dashboard_up():
    try:
        return urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2).read() == b"ok"
    except Exception:
        return False

if dashboard_up():
    print(f"Dashboard already running at http://localhost:{PORT}")
else:
    log = open("streamlit.log", "w")
    proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", str(PORT),
                             "--server.headless", "true"], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(30):
        time.sleep(1)
        if dashboard_up():
            break
    print(("Dashboard running" if dashboard_up() else "Dashboard still starting - see streamlit.log")
          + f" at http://localhost:{PORT}  (pid {proc.pid})")

In [ ]:
from IPython.display import IFrame
IFrame(f"http://localhost:{PORT}", width="100%", height=650)

## 16. Summary
* **Retrieval quality** (Text→Video, mean-pooled CLIP frames) and all other representations are reported in Section 8.
* **Research Investigation 1** – the data-driven answer is printed at the end of Section 9.
* **Deliverables:** this notebook (with outputs), the technical report (`artifacts/MIR_Assignment2_Technical_Report.docx`),
  and screenshots of the Streamlit dashboard running in the BITS Virtual Lab.